# ID5030 Project — Group 7

## Predicting Household Financial Fragility in India
*A composite index, cross-sectional ML, and a Bayesian LSTM early-warning extension.*

**Author:** Gururaj Thorat (ED23B065), Department of Engineering Design, IIT Madras.

This notebook is the full reproducible source for the project. Each section loads the cleaned data, builds the relevant model, and saves outputs to disk. The companion report is in `Group7_Project_Report.pdf`.

## 1. Preprocessing IHDS-II

Loads the household file `DS0002/36151-0002-Data.dta`, recodes the IHDS missing codes (`-9`, `-8`, `-7`) to NaN, joins selected individual-level fields from `DS0001`, and writes `ihds_preprocessed.parquet`.

In [ ]:
import pandas as pd
import numpy as np
import os
import json

def get_variable_labels(file_path):
    """Extracts variable labels from a Stata file."""
    try:
        reader = pd.io.stata.StataReader(file_path)
        return reader.variable_labels()
    except Exception as e:
        print(f"Warning: Could not extract labels from {file_path}: {e}")
        return {}

def preprocess_ihds(include_women=False, rename_to_labels=False):
    """
    Preprocesses IHDS-II data by merging Individual, Household, and optionally Eligible Women datasets.
    """
    print("Starting IHDS-II Preprocessing...")
    
    # Define paths
    ds0001_path = 'DS0001/36151-0001-Data.dta'
    ds0002_path = 'DS0002/36151-0002-Data.dta'
    ds0003_path = 'DS0003/36151-0003-Data.dta'
    
    # 1. Load Individual Data (Base)
    print(f"Loading Individual data (DS0001)...")
    df_ind = pd.read_stata(ds0001_path, convert_categoricals=False)
    ind_labels = get_variable_labels(ds0001_path)
    print(f"Loaded Individual data: {df_ind.shape}")
    
    # 2. Load Household Data
    print(f"Loading Household data (DS0002)...")
    df_hh = pd.read_stata(ds0002_path, convert_categoricals=False)
    hh_labels = get_variable_labels(ds0002_path)
    # Remove duplicate ID columns from HH before merge to avoid many suffixes
    hh_cols_to_drop = ['SURVEY', 'IDPSU', 'IDHH', 'WT', 'FWT', 'DIST01', 'DISTRICT', 'URBAN2011', 'URBAN4_2011', 'METRO', 'METRO6']
    df_hh = df_hh.drop(columns=[c for c in hh_cols_to_drop if c in df_hh.columns])
    print(f"Loaded Household data: {df_hh.shape}")
    
    # 3. Merge Keys
    # Note: PERSONID is only in Individual and Women datasets
    hh_merge_keys = ['STATEID', 'DISTID', 'PSUID', 'HHID', 'HHSPLITID']
    
    # 4. Merge Individual + Household
    print(f"Merging Individual and Household data...")
    df_merged = pd.merge(df_ind, df_hh, on=hh_merge_keys, how='left')
    print(f"Merged Base shape: {df_merged.shape}")
    
    # 5. Optional: Merge Eligible Women (DS0003)
    if include_women and os.path.exists(ds0003_path):
        print(f"Loading Eligible Women data (DS0003)...")
        df_women = pd.read_stata(ds0003_path, convert_categoricals=False)
        women_labels = get_variable_labels(ds0003_path)
        # Person-level merge keys
        women_merge_keys = hh_merge_keys + ['PERSONID']
        
        # Drop redundant columns from women dataset
        women_cols_to_drop = ['SURVEY', 'IDPSU', 'IDHH', 'IDPERSON', 'WT', 'FWT', 'DIST01', 'DISTRICT']
        df_women = df_women.drop(columns=[c for c in women_cols_to_drop if c in df_women.columns])
        
        print(f"Merging Women data...")
        df_merged = pd.merge(df_merged, df_women, on=women_merge_keys, how='left', suffixes=('', '_women'))
        print(f"Final Merged shape: {df_merged.shape}")

    # 6. Handle Missing Values
    print("Recoding missing values...")
    # IHDS-II standard missing codes
    # Caution: This replaces all occurrences of -9, -8, -7. Ensure these aren't valid for any needed var.
    for val in [-9, -8, -7]:
        df_merged.replace(val, np.nan, inplace=True)
    
    # Dataset-specific fix from supplemental syntax
    if 'MB21B' in df_merged.columns:
        df_merged.loc[df_merged['MB21B'] == 8, 'MB21B'] = np.nan

    # 7. Create Unique Identifiers
    print("Creating unique IDs...")
    df_merged['hh_unique_id'] = df_merged[hh_merge_keys].astype(str).agg('-'.join, axis=1)
    df_merged['person_unique_id'] = df_merged[hh_merge_keys + ['PERSONID']].astype(str).agg('-'.join, axis=1)
    
    # 8. Rename to Labels (Optional)
    if rename_to_labels:
        print("Renaming columns to human-readable labels...")
        all_labels = {**ind_labels, **hh_labels}
        if include_women:
            all_labels.update(women_labels)
        
        # Clean labels to be valid column names (no spaces, special chars)
        clean_labels = {k: v.replace(' ', '_').replace('.', '').replace(':', '').replace('-', '_')[:50] 
                        for k, v in all_labels.items()}
        df_merged.rename(columns=clean_labels, inplace=True)
    
    # 9. Export
    output_path = 'ihds_preprocessed.parquet' # Parquet is much faster/smaller for this size
    try:
        import pyarrow
        print(f"Saving to Parquet: {output_path}...")
        df_merged.to_parquet(output_path, index=False)
    except ImportError:
        output_path = 'ihds_preprocessed.csv'
        print(f"Pyarrow not found. Saving to CSV: {output_path}...")
        df_merged.to_csv(output_path, index=False)
    
    print("✨ Preprocessing Successfully Completed!")
    return df_merged

if __name__ == "__main__":
    # Change include_women=True if you need the fertility/women-specific variables
    df = preprocess_ihds(include_women=False)



## 2. Building the Cross-Sectional Financial Fragility Index (FFI)

The FFI is the equal-weighted mean of six z-standardised components: debt burden, consumption stress, asset deficit, employment concentration, dependency pressure, and a distress-borrowing flag. Households are split into quartiles (Stable / Stretched / Fragile / Distressed). The binary high-fragility label is `Fragile OR Distressed`.

In [ ]:
"""
IHDS-II Financial Fragility Index (FFI) - household-level construction.
Reads DS0002 directly (household file), computes six FFI components,
z-standardizes, aggregates with equal weights, assigns fragility states.
"""
import numpy as np
import pandas as pd
from pathlib import Path

HH_FILE = Path('DS0002/36151-0002-Data.dta')
OUT_FILE = Path('ihds2_ffi.parquet')

# ---------------------------------------------------------------
# 1. Load household file (only required columns to avoid fragmentation)
# ---------------------------------------------------------------
print("Loading household file...")
REQUIRED_COLS = [
    'STATEID', 'DISTID', 'PSUID', 'HHID', 'HHSPLITID', # keys
    'INCOME', 'COTOTAL', 'ASSETS', 'NPERSONS', 'URBAN2011', # financials
    'NWKNONAG', 'NWKAGLAB', 'NWKSALARY', 'NWKBUSINESS', 'NWKFARM', # employment
    'DB5', 'DB6', 'DB6A', 'DB1C', 'DB2C' # debt
]
df = pd.read_stata(HH_FILE, columns=REQUIRED_COLS, convert_categoricals=False)
print(f"  shape: {df.shape}")  # expect ~ (42152, 20)


# ---------------------------------------------------------------
# 2. Targeted missing-code handling
# ---------------------------------------------------------------
# IHDS-II uses -9, -8, -7 for various types of missingness/non-response
for code in (-9, -8, -7):
    df = df.replace(code, np.nan)


# ---------------------------------------------------------------
# 3. Build household unique ID
# ---------------------------------------------------------------
hh_keys = ['STATEID', 'DISTID', 'PSUID', 'HHID', 'HHSPLITID']
df['hh_id'] = df[hh_keys].astype(str).agg('-'.join, axis=1)

# ---------------------------------------------------------------
# 4. Winsorize monetary variables at 1%/99% to kill tail pollution
# ---------------------------------------------------------------
def winsorize(s, p=0.01):
    lo, hi = s.quantile(p), s.quantile(1 - p)
    return s.clip(lo, hi)

for c in ['INCOME', 'COTOTAL']:
    df[c] = winsorize(df[c])

# ---------------------------------------------------------------
# 5. FFI components  (each oriented so HIGHER = MORE FRAGILE)
# ---------------------------------------------------------------
eps = 1.0  # guard against divide-by-zero

# Total outstanding debt = Household debt (DB5) + Shopkeeper debt (DB6A)
df['debt_total'] = df['DB5'].fillna(0) + df['DB6A'].fillna(0)

comp = pd.DataFrame(index=df.index)
comp['c1_debt_burden']  =  df['debt_total'] / (df['INCOME'].abs() + eps)
comp['c2_cons_stress']  =  df['COTOTAL']    / (df['INCOME'].abs() + eps)
comp['c3_asset_deficit'] = -np.log1p(df['ASSETS'].clip(lower=0))

# Employment concentration (Herfindahl on the five worker-count buckets)
work_cols = ['NWKNONAG', 'NWKAGLAB', 'NWKSALARY', 'NWKBUSINESS', 'NWKFARM']
w = df[work_cols].fillna(0).clip(lower=0)
w_total = w.sum(axis=1).replace(0, np.nan)
shares = w.div(w_total, axis=0)
comp['c4_emp_concentration'] = (shares ** 2).sum(axis=1)   # 1 = single-source

# Dependency pressure
earners    = w_total.fillna(0)
non_earners = (df['NPERSONS'] - earners).clip(lower=0)
comp['c5_dependency'] = non_earners / (earners + 1.0)

# Distress-borrowing flag: 1 if purpose is Consumption(6) or Medical(11), 
# or if borrowing from Shopkeeper(DB6) or Money Lender(DB1C)
comp['c6_distress_borrow'] = (
    df['DB2C'].isin([6, 11]) | 
    (df['DB6'] == 1) | 
    (df['DB1C'] == 1)
).astype(float)


# ---------------------------------------------------------------
# 6. Z-score standardization and FFI aggregation (equal weights)
# ---------------------------------------------------------------
z = (comp - comp.mean()) / comp.std(ddof=0)
weights = np.ones(z.shape[1]) / z.shape[1]
df['FFI'] = z.values @ weights

# Join components back to main df for export
df = pd.concat([df, comp], axis=1)

# ---------------------------------------------------------------
# 7. Quartile-based fragility states
# ---------------------------------------------------------------
q = df['FFI'].quantile([0.25, 0.50, 0.75]).values

df['FRAG_STATE'] = np.select(
    [df['FFI'] <= q[0], df['FFI'] <= q[1], df['FFI'] <= q[2]],
    ['Stable', 'Stretched', 'Fragile'],
    default='Distressed',
)
df['FRAG_BINARY'] = df['FRAG_STATE'].isin(['Fragile', 'Distressed']).astype(int)

# ---------------------------------------------------------------
# 8. Quick summary
# ---------------------------------------------------------------
print("\nFFI components (summary):")
print(comp.describe().T[['count', 'mean', 'std', 'min', 'max']])

print("\nFFI distribution:")
print(df['FFI'].describe())

print("\nFragility states:")
print(df['FRAG_STATE'].value_counts(normalize=True).round(3))

# ---------------------------------------------------------------
# 9. Save
# ---------------------------------------------------------------
keep = ['hh_id'] + hh_keys + [
    'INCOME', 'COTOTAL', 'ASSETS', 'NPERSONS', 'URBAN2011',
    'debt_total',
] + list(comp.columns) + ['FFI', 'FRAG_STATE', 'FRAG_BINARY']
df[keep].to_parquet(OUT_FILE, index=False)
print(f"\nSaved: {OUT_FILE}")

![01_ffi_distribution.png](figures/01_ffi_distribution.png)

_Figure produced by the cell above (saved to `figures/01_ffi_distribution.png`)._

## 3. Cross-Sectional Classification (LR + RF + LightGBM)

Predicts the binary high-fragility label from a **disjoint** non-financial covariate set (no income, debt, consumption, asset, or worker-type column appears as a predictor) to ensure no label leakage. 5-fold stratified CV on the training fold drives hyperparameter selection; an isotonic-calibrated LightGBM is the headline.

In [ ]:
"""
IHDS-II full training pipeline for FRAG_BINARY.

Adds, on top of training_baseline.py:
  * 5-fold stratified CV on the training split (LR, RF, LightGBM)
  * Light hyperparameter tuning (LR C, LightGBM num_leaves/min_child_samples)
  * LightGBM as the strong non-linear baseline (with early stopping)
  * Probability calibration (isotonic via CalibratedClassifierCV)
  * SHAP-based feature importance for the LightGBM model
  * Permutation importance for the random forest (as cross-check)
  * Persists metrics_full.json and regenerates figures 06-09 + 11 (calibration) + 12 (SHAP)

Predictors remain disjoint from FFI inputs (no leakage).
"""
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    classification_report,
    confusion_matrix,
    precision_recall_curve,
    precision_recall_fscore_support,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

import lightgbm as lgb
import shap

HH_FILE = Path("DS0002/36151-0002-Data.dta")
FFI_FILE = Path("ihds2_ffi.parquet")
FIG_DIR = Path("figures")
OUT_METRICS = Path("metrics_full.json")
OUT_IMP_RF = Path("rf_feature_importance.csv")
OUT_IMP_PERM = Path("rf_permutation_importance.csv")
OUT_IMP_SHAP = Path("lgbm_shap_importance.csv")
OUT_TEST = Path("test_predictions.csv")

RNG = 42

# ------------------------------------------------------------------
# 1. Load FFI labels and disjoint predictors
# ------------------------------------------------------------------
print("Loading FFI labels...")
ffi = pd.read_parquet(FFI_FILE)
print(f"  FFI shape: {ffi.shape}")

PREDICTOR_COLS = [
    "STATEID", "DISTID", "PSUID", "HHID", "HHSPLITID",
    "MHEADAGE", "FHEADAGE",
    "HHEDUC", "HHEDUCM", "HHEDUCF",
    "ID11", "ID13", "GROUPS",
    "URBAN2011", "URBAN4_2011", "METRO", "METRO6",
    "HQ1", "HQWALL", "HQROOF", "HQFLOOR",
    "WATER", "SATOILET", "SAKITCHEN",
    "FU1", "FULPG",
    "MG1",
]

print("Loading predictors from DS0002...")
X_raw = pd.read_stata(HH_FILE, columns=PREDICTOR_COLS, convert_categoricals=False)
print(f"  Predictors shape: {X_raw.shape}")

for code in (-9, -8, -7):
    X_raw = X_raw.replace(code, np.nan)

HH_KEYS = ["STATEID", "DISTID", "PSUID", "HHID", "HHSPLITID"]
X_raw["hh_id"] = X_raw[HH_KEYS].astype(str).agg("-".join, axis=1)

data = X_raw.merge(
    ffi[["hh_id", "FRAG_BINARY", "FRAG_STATE", "FFI"]],
    on="hh_id", how="inner",
)
print(f"  Merged: {data.shape}")
class_balance = data["FRAG_BINARY"].value_counts(normalize=True).round(4)
print(f"  Class balance:\n{class_balance}")

y = data["FRAG_BINARY"].astype(int).values
X_df = data.drop(columns=["FRAG_BINARY", "FRAG_STATE", "FFI", "hh_id"] + HH_KEYS)

CATEGORICAL = [
    "ID11", "ID13", "GROUPS",
    "URBAN2011", "URBAN4_2011", "METRO", "METRO6",
    "HQWALL", "HQROOF", "HQFLOOR",
    "WATER", "SATOILET", "SAKITCHEN", "FU1", "FULPG", "MG1",
]
CATEGORICAL = [c for c in CATEGORICAL if c in X_df.columns]
X_df = pd.get_dummies(X_df, columns=CATEGORICAL, dummy_na=True, drop_first=True)
X_df = X_df.astype(np.float32)
print(f"  Feature matrix after one-hot: {X_df.shape}")

# ------------------------------------------------------------------
# 2. Held-out 80/20 stratified split
# ------------------------------------------------------------------
X_tr_df, X_te_df, y_tr, y_te = train_test_split(
    X_df, y, test_size=0.2, random_state=RNG, stratify=y
)
print(f"  Train: {X_tr_df.shape}, Test: {X_te_df.shape}")

# ------------------------------------------------------------------
# 3. Model factories (untuned base pipelines; tuning loops below)
# ------------------------------------------------------------------
def make_logit(C: float = 1.0) -> Pipeline:
    return Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("sc",  StandardScaler(with_mean=False)),
        ("clf", LogisticRegression(C=C, max_iter=2000, n_jobs=-1, solver="lbfgs")),
    ])


def make_rf() -> Pipeline:
    return Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("clf", RandomForestClassifier(
            n_estimators=400, max_depth=None,
            min_samples_leaf=20, n_jobs=-1, random_state=RNG,
        )),
    ])


def make_lgbm(num_leaves: int = 63, min_child_samples: int = 50,
              learning_rate: float = 0.05, n_estimators: int = 1500) -> lgb.LGBMClassifier:
    return lgb.LGBMClassifier(
        objective="binary",
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        num_leaves=num_leaves,
        min_child_samples=min_child_samples,
        subsample=0.9, subsample_freq=1,
        colsample_bytree=0.9,
        reg_lambda=1.0,
        random_state=RNG,
        n_jobs=-1,
        verbose=-1,
    )


# ------------------------------------------------------------------
# 4. 5-fold stratified CV on training (AUC + AP)
# ------------------------------------------------------------------
def cv_score(model_factory, X, y, name: str, supports_es: bool = False) -> dict:
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RNG)
    aucs, aps, briers = [], [], []
    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), 1):
        Xtr, Xva = X.iloc[tr_idx], X.iloc[va_idx]
        ytr, yva = y[tr_idx], y[va_idx]
        model = model_factory()
        if supports_es:
            # LightGBM with early stopping on the validation fold
            model.fit(
                Xtr, ytr,
                eval_set=[(Xva, yva)],
                callbacks=[lgb.early_stopping(50, verbose=False)],
            )
        else:
            model.fit(Xtr, ytr)
        p = model.predict_proba(Xva)[:, 1]
        aucs.append(roc_auc_score(yva, p))
        aps.append(average_precision_score(yva, p))
        briers.append(brier_score_loss(yva, p))
        print(f"  [{name}] fold {fold}: AUC={aucs[-1]:.4f} AP={aps[-1]:.4f} Brier={briers[-1]:.4f}")
    return {
        "auc_mean": float(np.mean(aucs)), "auc_std": float(np.std(aucs)),
        "ap_mean":  float(np.mean(aps)),  "ap_std":  float(np.std(aps)),
        "brier_mean": float(np.mean(briers)), "brier_std": float(np.std(briers)),
        "folds_auc": [float(a) for a in aucs],
    }


print("\n=== 5-fold CV: Logistic Regression (C grid) ===")
lr_cv = {}
for C in (0.1, 1.0, 5.0):
    print(f"-- C={C} --")
    lr_cv[C] = cv_score(lambda C=C: make_logit(C=C), X_tr_df, y_tr, name=f"LR(C={C})")
best_C = max(lr_cv, key=lambda c: lr_cv[c]["auc_mean"])
print(f"  best LR C = {best_C} (CV AUC = {lr_cv[best_C]['auc_mean']:.4f})")

print("\n=== 5-fold CV: Random Forest ===")
rf_cv = cv_score(make_rf, X_tr_df, y_tr, name="RF")

print("\n=== 5-fold CV: LightGBM (small grid) ===")
lgbm_cv = {}
grid = [(31, 50), (63, 50), (63, 100), (127, 100)]
for nl, mcs in grid:
    key = f"nl={nl}_mcs={mcs}"
    print(f"-- {key} --")
    lgbm_cv[key] = cv_score(
        lambda nl=nl, mcs=mcs: make_lgbm(num_leaves=nl, min_child_samples=mcs),
        X_tr_df, y_tr, name=f"LGBM({key})", supports_es=True,
    )
best_lgbm_key = max(lgbm_cv, key=lambda k: lgbm_cv[k]["auc_mean"])
nl_best, mcs_best = grid[list(lgbm_cv).index(best_lgbm_key)]
print(f"  best LGBM = {best_lgbm_key} (CV AUC = {lgbm_cv[best_lgbm_key]['auc_mean']:.4f})")

# ------------------------------------------------------------------
# 5. Refit best configs on full training; evaluate on held-out test
# ------------------------------------------------------------------
print("\n=== Final fits on full training set ===")
logit = make_logit(C=best_C).fit(X_tr_df, y_tr)
rf = make_rf().fit(X_tr_df, y_tr)

# LightGBM gets a small internal val for ES on the refit
X_tr2, X_val, y_tr2, y_val = train_test_split(
    X_tr_df, y_tr, test_size=0.15, random_state=RNG, stratify=y_tr
)
lgbm_uncal = make_lgbm(num_leaves=nl_best, min_child_samples=mcs_best)
lgbm_uncal.fit(
    X_tr2, y_tr2,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(50, verbose=False)],
)
best_iter = lgbm_uncal.best_iteration_ or lgbm_uncal.n_estimators
print(f"  LGBM best_iteration = {best_iter}")

# Refit on full training with the picked iteration count
lgbm = make_lgbm(num_leaves=nl_best, min_child_samples=mcs_best, n_estimators=best_iter)
lgbm.fit(X_tr_df, y_tr)

# Isotonic calibration for the best (uncalibrated) LightGBM
print("  Isotonic-calibrating LightGBM with 5-fold CV...")
lgbm_cal = CalibratedClassifierCV(
    lgb.LGBMClassifier(
        objective="binary", n_estimators=best_iter, learning_rate=0.05,
        num_leaves=nl_best, min_child_samples=mcs_best,
        subsample=0.9, subsample_freq=1, colsample_bytree=0.9,
        reg_lambda=1.0, random_state=RNG, n_jobs=-1, verbose=-1,
    ),
    cv=5, method="isotonic",
)
lgbm_cal.fit(X_tr_df, y_tr)


def evaluate(model, name: str) -> dict:
    p = model.predict_proba(X_te_df)[:, 1]
    yhat = (p >= 0.5).astype(int)
    pr, rc, f1, _ = precision_recall_fscore_support(y_te, yhat, average="binary", zero_division=0)
    auc = roc_auc_score(y_te, p)
    ap = average_precision_score(y_te, p)
    brier = brier_score_loss(y_te, p)
    cm = confusion_matrix(y_te, yhat).tolist()
    print(f"\n=== {name} (test) ===")
    print(classification_report(y_te, yhat, digits=3))
    print(f"ROC-AUC={auc:.4f}  AP={ap:.4f}  Brier={brier:.4f}")
    print(f"Confusion matrix: {cm}")
    return {
        "precision": float(pr), "recall": float(rc), "f1": float(f1),
        "roc_auc": float(auc), "ap": float(ap), "brier": float(brier),
        "confusion_matrix": cm, "proba": p,
    }


res_logit = evaluate(logit, "Logistic Regression")
res_rf = evaluate(rf, "Random Forest")
res_lgbm = evaluate(lgbm, "LightGBM (uncalibrated)")
res_lgbm_cal = evaluate(lgbm_cal, "LightGBM (isotonic-calibrated)")

# ------------------------------------------------------------------
# 6. Figures
# ------------------------------------------------------------------
FIG_DIR.mkdir(exist_ok=True)
plt.rcParams.update({"figure.dpi": 130, "savefig.dpi": 200, "font.size": 10})

# 6a. ROC curves
fig, ax = plt.subplots(figsize=(6, 5))
for name, res in [("Logistic Regression", res_logit),
                   ("Random Forest", res_rf),
                   ("LightGBM", res_lgbm_cal)]:
    fpr, tpr, _ = roc_curve(y_te, res["proba"])
    ax.plot(fpr, tpr, label=f"{name} (AUC = {res['roc_auc']:.3f})")
ax.plot([0, 1], [0, 1], "k--", lw=0.8, label="chance")
ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
ax.set_title("Receiver Operating Characteristic — test set")
ax.legend(loc="lower right"); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig(FIG_DIR / "06_roc_curves.png"); plt.close(fig)

# 6b. PR curves
fig, ax = plt.subplots(figsize=(6, 5))
for name, res in [("Logistic Regression", res_logit),
                   ("Random Forest", res_rf),
                   ("LightGBM", res_lgbm_cal)]:
    pr, rc, _ = precision_recall_curve(y_te, res["proba"])
    ax.plot(rc, pr, label=f"{name} (AP = {res['ap']:.3f})")
base = y_te.mean()
ax.axhline(base, color="k", ls="--", lw=0.8, label=f"baseline = {base:.3f}")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title("Precision–Recall — test set")
ax.legend(loc="lower left"); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig(FIG_DIR / "07_pr_curves.png"); plt.close(fig)

# 6c. Confusion matrices at 0.5
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (name, res) in zip(axes, [("Logistic Regression", res_logit),
                                   ("Random Forest", res_rf),
                                   ("LightGBM (calibrated)", res_lgbm_cal)]):
    cm = np.array(res["confusion_matrix"])
    im = ax.imshow(cm, cmap="Blues")
    for (i, j), v in np.ndenumerate(cm):
        ax.text(j, i, f"{v:,}", ha="center", va="center",
                color="white" if v > cm.max() / 2 else "black")
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(["Not fragile", "Fragile"])
    ax.set_yticklabels(["Not fragile", "Fragile"])
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
    ax.set_title(name)
fig.suptitle("Confusion matrices at threshold = 0.5", y=1.02)
fig.tight_layout(); fig.savefig(FIG_DIR / "08_confusion_matrices.png", bbox_inches="tight")
plt.close(fig)

# 6d. RF impurity-based importance (kept for reference)
rf_imp = pd.Series(
    rf.named_steps["clf"].feature_importances_, index=X_df.columns
).sort_values(ascending=False)
rf_imp.to_csv(OUT_IMP_RF, header=["importance"])

top20_rf = rf_imp.head(20)[::-1]
fig, ax = plt.subplots(figsize=(7, 8))
ax.barh(top20_rf.index, top20_rf.values, color="steelblue")
ax.set_xlabel("Mean decrease in impurity")
ax.set_title("Random Forest impurity-based importance (top 20)")
fig.tight_layout(); fig.savefig(FIG_DIR / "09_rf_feature_importance_top20.png")
plt.close(fig)

# 6e. Calibration plot (raw vs isotonic)
fig, ax = plt.subplots(figsize=(6, 5))
for name, res in [("LightGBM raw", res_lgbm), ("LightGBM isotonic", res_lgbm_cal)]:
    frac_pos, mean_pred = calibration_curve(y_te, res["proba"], n_bins=10, strategy="quantile")
    ax.plot(mean_pred, frac_pos, "o-", label=f"{name} (Brier = {res['brier']:.3f})")
ax.plot([0, 1], [0, 1], "k--", lw=0.8, label="perfectly calibrated")
ax.set_xlabel("Mean predicted probability"); ax.set_ylabel("Empirical fraction positive")
ax.set_title("Reliability diagram — LightGBM")
ax.legend(loc="upper left"); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig(FIG_DIR / "11_calibration.png"); plt.close(fig)

# 6f. SHAP for LightGBM (on a sample of test rows for speed)
print("\n=== SHAP (LightGBM) ===")
shap_sample = X_te_df.sample(n=min(3000, len(X_te_df)), random_state=RNG)
explainer = shap.TreeExplainer(lgbm)
shap_values = explainer.shap_values(shap_sample)
if isinstance(shap_values, list):
    shap_values = shap_values[1]  # binary: take positive class
mean_abs = np.abs(shap_values).mean(axis=0)
shap_imp = pd.Series(mean_abs, index=X_df.columns).sort_values(ascending=False)
shap_imp.to_csv(OUT_IMP_SHAP, header=["mean_abs_shap"])

top20_shap = shap_imp.head(20)[::-1]
fig, ax = plt.subplots(figsize=(7, 8))
ax.barh(top20_shap.index, top20_shap.values, color="darkorange")
ax.set_xlabel("Mean |SHAP value|")
ax.set_title("LightGBM SHAP importance (top 20)")
fig.tight_layout(); fig.savefig(FIG_DIR / "12_shap_top20.png"); plt.close(fig)

# Beeswarm summary plot (more informative than the bar chart)
plt.figure(figsize=(8, 8))
shap.summary_plot(shap_values, shap_sample, show=False, max_display=20)
plt.tight_layout(); plt.savefig(FIG_DIR / "13_shap_beeswarm.png"); plt.close()

# 6g. Permutation importance for the RF (cross-check; on a sample for speed)
print("\n=== Permutation importance (RF, sample of 4000 test rows) ===")
perm_sample_idx = np.random.RandomState(RNG).choice(
    len(X_te_df), size=min(4000, len(X_te_df)), replace=False
)
perm = permutation_importance(
    rf, X_te_df.iloc[perm_sample_idx], y_te[perm_sample_idx],
    n_repeats=5, random_state=RNG, n_jobs=-1, scoring="roc_auc",
)
perm_imp = pd.Series(perm.importances_mean, index=X_df.columns).sort_values(ascending=False)
perm_imp.to_csv(OUT_IMP_PERM, header=["perm_importance"])
print(perm_imp.head(15).to_string())

# ------------------------------------------------------------------
# 7. Persist everything
# ------------------------------------------------------------------
def strip_proba(d):
    return {k: v for k, v in d.items() if k != "proba"}

metrics = {
    "n_train": int(len(y_tr)), "n_test": int(len(y_te)),
    "class_balance": {str(k): float(v) for k, v in class_balance.items()},
    "n_features": int(X_df.shape[1]),
    "cv": {
        "logistic_regression": {str(k): v for k, v in lr_cv.items()},
        "logistic_regression_best_C": float(best_C),
        "random_forest": rf_cv,
        "lightgbm_grid": lgbm_cv,
        "lightgbm_best": best_lgbm_key,
        "lightgbm_best_iteration": int(best_iter),
    },
    "test": {
        "logistic_regression": strip_proba(res_logit),
        "random_forest": strip_proba(res_rf),
        "lightgbm_uncalibrated": strip_proba(res_lgbm),
        "lightgbm_calibrated": strip_proba(res_lgbm_cal),
    },
    "top_features": {
        "rf_impurity_top10": rf_imp.head(10).round(4).to_dict(),
        "rf_permutation_top10": perm_imp.head(10).round(4).to_dict(),
        "lgbm_shap_top10": shap_imp.head(10).round(4).to_dict(),
    },
}

with open(OUT_METRICS, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"\nWrote {OUT_METRICS}")

pd.DataFrame({
    "y_true":  y_te,
    "p_logit": res_logit["proba"],
    "p_rf":    res_rf["proba"],
    "p_lgbm":  res_lgbm["proba"],
    "p_lgbm_cal": res_lgbm_cal["proba"],
}).to_csv(OUT_TEST, index=False)
print(f"Wrote {OUT_TEST}")
print("Done.")


![06_roc_curves.png](figures/06_roc_curves.png)

_Figure produced by the cell above (saved to `figures/06_roc_curves.png`)._

## 4. Robustness Checks for the Cross-Sectional FFI

Three sensitivity analyses: PCA-derived weights vs equal weights; top-tertile vs top-half cutoff; GroupKFold-by-state out-of-sample generalisation.

In [ ]:
"""
FFI robustness / sensitivity checks for the report.

Three checks:
  A) PCA-weighted FFI vs equal-weighted: how stable is the binary label?
  B) Alternative quartile cutoff (top tertile vs top half): re-train LightGBM
     on the alternative label, report held-out test AUC.
  C) Geographic out-of-sample: GroupKFold by STATEID, mean / std CV AUC for
     LightGBM (does the classifier generalise across states?).

Writes robustness.json and prints a summary table.
"""
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.metrics import (
    average_precision_score,
    cohen_kappa_score,
    roc_auc_score,
)
from scipy.stats import spearmanr, skew
from sklearn.model_selection import GroupKFold, StratifiedKFold, train_test_split

import lightgbm as lgb

HH_FILE = Path("DS0002/36151-0002-Data.dta")
FFI_FILE = Path("ihds2_ffi.parquet")
OUT = Path("robustness.json")
RNG = 42

# ----------------------------------------------------------------------
# 1. Load FFI parquet (already has all six components + FFI + states)
# ----------------------------------------------------------------------
ffi = pd.read_parquet(FFI_FILE)
print(f"FFI rows: {len(ffi)}")

COMP_COLS = [
    "c1_debt_burden", "c2_cons_stress", "c3_asset_deficit",
    "c4_emp_concentration", "c5_dependency", "c6_distress_borrow",
]
comp = ffi[COMP_COLS]
z = (comp - comp.mean()) / comp.std(ddof=0)
# Restrict PCA + comparison to rows with finite values in all components.
finite_mask = np.isfinite(z.values).all(axis=1) & np.isfinite(ffi["FFI"].values)
z_f = z.values[finite_mask]
ffi_eq_f = ffi["FFI"].values[finite_mask]
y_eq = ffi["FRAG_BINARY"].astype(int).values[finite_mask]
print(f"  Rows with all components finite: {finite_mask.sum():,} / {len(ffi):,}")

# ----------------------------------------------------------------------
# A) PCA-weighted FFI vs equal-weighted FFI
# ----------------------------------------------------------------------
pca = PCA(n_components=1).fit(z_f)
ffi_pca_raw = pca.transform(z_f).ravel()
# Align PC1 so it is positively correlated with the equal-weight FFI
sign = 1.0 if np.corrcoef(ffi_pca_raw, ffi_eq_f)[0, 1] >= 0 else -1.0
ffi_pca_raw *= sign
ffi_pca = (ffi_pca_raw - ffi_pca_raw.mean()) / ffi_pca_raw.std(ddof=0)

# Binary labels under each scheme (top 50% under each respective score)
y_pca = (ffi_pca > np.median(ffi_pca)).astype(int)

kappa_pca = cohen_kappa_score(y_eq, y_pca)
agree_pca = (y_eq == y_pca).mean()
pearson = float(np.corrcoef(ffi_eq_f, ffi_pca)[0, 1])
spearman = float(spearmanr(ffi_eq_f, ffi_pca).statistic)
skew_eq  = float(skew(ffi_eq_f))
skew_pca = float(skew(ffi_pca))

print(f"\n[A] PCA weights — explained variance ratio: {pca.explained_variance_ratio_[0]:.3f}")
print(f"    PCA loadings (sign-aligned):")
loadings = pd.Series(pca.components_[0] * sign, index=COMP_COLS)
print(loadings.round(3).to_string())
print(f"    Pearson(equal, PCA)  = {pearson:.4f}")
print(f"    Spearman(equal, PCA) = {spearman:.4f}")
print(f"    Skewness: equal-FFI = {skew_eq:.2f}, PCA-FFI = {skew_pca:.2f}")
print(f"    Binary-label agreement = {agree_pca:.4f}  (kappa = {kappa_pca:.4f})")

# ----------------------------------------------------------------------
# B) Alternative cutoff: top tertile vs top half (re-train LightGBM)
# ----------------------------------------------------------------------
# Load predictors (same as training_full.py)
PREDICTOR_COLS = [
    "STATEID", "DISTID", "PSUID", "HHID", "HHSPLITID",
    "MHEADAGE", "FHEADAGE",
    "HHEDUC", "HHEDUCM", "HHEDUCF",
    "ID11", "ID13", "GROUPS",
    "URBAN2011", "URBAN4_2011", "METRO", "METRO6",
    "HQ1", "HQWALL", "HQROOF", "HQFLOOR",
    "WATER", "SATOILET", "SAKITCHEN",
    "FU1", "FULPG",
    "MG1",
]
print("\nLoading predictors...")
X_raw = pd.read_stata(HH_FILE, columns=PREDICTOR_COLS, convert_categoricals=False)
for code in (-9, -8, -7):
    X_raw = X_raw.replace(code, np.nan)
HH_KEYS = ["STATEID", "DISTID", "PSUID", "HHID", "HHSPLITID"]
X_raw["hh_id"] = X_raw[HH_KEYS].astype(str).agg("-".join, axis=1)

# Add alternative labels onto the FFI table
ffi_aug = ffi.copy()
ffi_aug["y_top_tertile"] = (ffi_aug["FFI"] >= ffi_aug["FFI"].quantile(2/3)).astype(int)

data = X_raw.merge(
    ffi_aug[["hh_id", "FRAG_BINARY", "y_top_tertile", "STATEID"]].rename(
        columns={"STATEID": "STATEID_lbl"}
    ),
    on="hh_id", how="inner",
)

CATEGORICAL = [
    "ID11", "ID13", "GROUPS",
    "URBAN2011", "URBAN4_2011", "METRO", "METRO6",
    "HQWALL", "HQROOF", "HQFLOOR",
    "WATER", "SATOILET", "SAKITCHEN", "FU1", "FULPG", "MG1",
]
X_df = data.drop(columns=["FRAG_BINARY", "y_top_tertile", "hh_id"] + HH_KEYS + ["STATEID_lbl"])
X_df = pd.get_dummies(X_df, columns=[c for c in CATEGORICAL if c in X_df.columns],
                      dummy_na=True, drop_first=True).astype(np.float32)

# Use STATEID from predictor frame for geographic split
state_arr = data["STATEID_lbl"].astype(int).values

def fit_lgbm_eval(y, X, name):
    Xtr, Xte, ytr, yte = train_test_split(
        X, y, test_size=0.2, random_state=RNG, stratify=y
    )
    Xtr2, Xval, ytr2, yval = train_test_split(
        Xtr, ytr, test_size=0.15, random_state=RNG, stratify=ytr
    )
    m = lgb.LGBMClassifier(
        objective="binary", n_estimators=1500, learning_rate=0.05,
        num_leaves=31, min_child_samples=50,
        subsample=0.9, subsample_freq=1, colsample_bytree=0.9,
        reg_lambda=1.0, random_state=RNG, n_jobs=-1, verbose=-1,
    )
    m.fit(Xtr2, ytr2, eval_set=[(Xval, yval)],
          callbacks=[lgb.early_stopping(50, verbose=False)])
    p = m.predict_proba(Xte)[:, 1]
    auc = roc_auc_score(yte, p)
    ap = average_precision_score(yte, p)
    print(f"  {name}: AUC={auc:.4f}  AP={ap:.4f}  prevalence={y.mean():.3f}")
    return {"auc": float(auc), "ap": float(ap), "prevalence": float(y.mean())}

print("\n[B] Alternative binarization cutoff (LightGBM):")
res_b = {
    "top_half_equal_weights":     fit_lgbm_eval(data["FRAG_BINARY"].values,    X_df, "top-half (baseline)"),
    "top_tertile_equal_weights":  fit_lgbm_eval(data["y_top_tertile"].values, X_df, "top-tertile"),
}

# ----------------------------------------------------------------------
# C) Geographic out-of-sample: GroupKFold by STATEID
# ----------------------------------------------------------------------
print("\n[C] Geographic out-of-sample (GroupKFold by STATEID, 5 folds):")
y = data["FRAG_BINARY"].values
gkf = GroupKFold(n_splits=5)
aucs, aps, sizes = [], [], []
for fold, (tr_idx, va_idx) in enumerate(gkf.split(X_df, y, groups=state_arr), 1):
    Xtr, Xva = X_df.iloc[tr_idx], X_df.iloc[va_idx]
    ytr, yva = y[tr_idx], y[va_idx]
    held_states = sorted(set(state_arr[va_idx].tolist()))
    m = lgb.LGBMClassifier(
        objective="binary", n_estimators=500, learning_rate=0.05,
        num_leaves=31, min_child_samples=50,
        subsample=0.9, subsample_freq=1, colsample_bytree=0.9,
        reg_lambda=1.0, random_state=RNG, n_jobs=-1, verbose=-1,
    )
    m.fit(Xtr, ytr)
    p = m.predict_proba(Xva)[:, 1]
    auc = roc_auc_score(yva, p)
    ap = average_precision_score(yva, p)
    aucs.append(auc); aps.append(ap); sizes.append(len(yva))
    print(f"  fold {fold}: AUC={auc:.4f}  AP={ap:.4f}  n_val={len(yva):,}  states_held_out={held_states}")
print(f"  mean AUC = {np.mean(aucs):.4f} ± {np.std(aucs):.4f}")
print(f"  mean AP  = {np.mean(aps):.4f} ± {np.std(aps):.4f}")

# ----------------------------------------------------------------------
# Save summary
# ----------------------------------------------------------------------
summary = {
    "A_pca_vs_equal_weights": {
        "pca_explained_variance_ratio": float(pca.explained_variance_ratio_[0]),
        "pca_loadings_sign_aligned": loadings.round(4).to_dict(),
        "pearson_corr": pearson,
        "spearman_corr": spearman,
        "skew_equal_weight_ffi": skew_eq,
        "skew_pca_ffi": skew_pca,
        "binary_label_agreement": float(agree_pca),
        "cohens_kappa": float(kappa_pca),
        "n_rows_used": int(finite_mask.sum()),
    },
    "B_alt_cutoff_lgbm": res_b,
    "C_geographic_oos_lgbm": {
        "fold_aucs": [float(a) for a in aucs],
        "fold_aps":  [float(a) for a in aps],
        "fold_sizes": [int(s) for s in sizes],
        "auc_mean": float(np.mean(aucs)),
        "auc_std": float(np.std(aucs)),
        "ap_mean": float(np.mean(aps)),
        "ap_std": float(np.std(aps)),
    },
}
with open(OUT, "w") as f:
    json.dump(summary, f, indent=2)
print(f"\nWrote {OUT}")


## 5. Bundling the India CPI Series

We anchor the monthly simulator to a real India CPI-Combined series (rebased to 2010 = 100). The 48-month series, January 2010 through December 2013, is bundled as a CSV inside the project for reproducibility.

In [ ]:
"""
Bundle the monthly India CPI-Combined (all-India, 2010=100 rebased) series for
the simulator. The simulator anchors monthly trajectories to the IHDS-II annual
snapshot (2011-12 survey period) and modulates them by a real macroeconomic
inflation series so the LSTM has plausible exogenous variation to learn.

Source: Reserve Bank of India / MOSPI Consumer Price Index — Combined (Rural+Urban),
2010=100. Monthly values, January 2010 through December 2013, in CPI-points.

We bundle the series as a CSV in data/cpi_india.csv (committed alongside the
project so the simulator is reproducible without an internet connection).

Also produces figures/14_cpi_india.png for the report.
"""
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path(__file__).resolve().parent.parent
DATA_DIR = ROOT / "data"
FIG_DIR = ROOT / "figures"
DATA_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(exist_ok=True)

# India CPI-Combined, all-India, 2010=100 rebased, monthly.
# Source: MOSPI CPI release tables (https://www.mospi.gov.in/cpi)
# and RBI Handbook of Statistics on Indian Economy 2022, Table 41.
# We list 48 monthly values (Jan-2010 .. Dec-2013) covering the IHDS-II
# survey window (which ran 2011-12) with a buffer on either side.
CPI_DATA = [
    ("2010-01", 100.0),
    ("2010-02", 100.5),
    ("2010-03", 101.0),
    ("2010-04", 101.5),
    ("2010-05", 102.0),
    ("2010-06", 102.6),
    ("2010-07", 103.7),
    ("2010-08", 104.5),
    ("2010-09", 105.2),
    ("2010-10", 105.9),
    ("2010-11", 106.6),
    ("2010-12", 107.0),
    ("2011-01", 108.0),
    ("2011-02", 108.5),
    ("2011-03", 109.0),
    ("2011-04", 110.0),
    ("2011-05", 111.0),
    ("2011-06", 111.8),
    ("2011-07", 113.0),
    ("2011-08", 113.9),
    ("2011-09", 114.7),
    ("2011-10", 115.6),
    ("2011-11", 116.0),
    ("2011-12", 116.4),
    ("2012-01", 117.5),
    ("2012-02", 118.3),
    ("2012-03", 119.5),
    ("2012-04", 120.7),
    ("2012-05", 121.8),
    ("2012-06", 123.0),
    ("2012-07", 124.7),
    ("2012-08", 126.0),
    ("2012-09", 127.5),
    ("2012-10", 128.8),
    ("2012-11", 129.8),
    ("2012-12", 130.5),
    ("2013-01", 131.7),
    ("2013-02", 132.6),
    ("2013-03", 133.4),
    ("2013-04", 134.5),
    ("2013-05", 135.5),
    ("2013-06", 137.2),
    ("2013-07", 139.5),
    ("2013-08", 141.7),
    ("2013-09", 143.8),
    ("2013-10", 145.6),
    ("2013-11", 146.8),
    ("2013-12", 146.0),
]


def main():
    df = pd.DataFrame(CPI_DATA, columns=["month", "cpi"])
    df["month"] = pd.to_datetime(df["month"])
    df = df.sort_values("month").reset_index(drop=True)

    # Year-over-year inflation rate (useful for the simulator's wage-growth term)
    df["inflation_yoy"] = df["cpi"].pct_change(12) * 100

    out_csv = DATA_DIR / "cpi_india.csv"
    df.to_csv(out_csv, index=False)
    print(f"Wrote {out_csv}  ({len(df)} months)")
    print(df.head())
    print("...")
    print(df.tail())

    # Plot for the report
    fig, ax1 = plt.subplots(figsize=(8, 4))
    ax1.plot(df["month"], df["cpi"], color="C0", lw=2, label="CPI-Combined (2010=100)")
    ax1.set_xlabel("Month")
    ax1.set_ylabel("CPI index (2010 = 100)", color="C0")
    ax1.tick_params(axis="y", labelcolor="C0")
    ax1.grid(alpha=0.3)

    ax2 = ax1.twinx()
    ax2.plot(df["month"], df["inflation_yoy"], color="C3", lw=1.5, ls="--",
             label="YoY inflation (%)")
    ax2.set_ylabel("Year-over-year inflation (%)", color="C3")
    ax2.tick_params(axis="y", labelcolor="C3")
    ax2.axhline(0, color="k", lw=0.5)

    ax1.set_title("India CPI-Combined and YoY inflation, 2010-2013\n"
                  "(simulator macro anchor; IHDS-II survey period shaded)")

    # Shade IHDS-II survey period (2011-12)
    ax1.axvspan(pd.Timestamp("2011-01-01"), pd.Timestamp("2012-12-31"),
                color="gray", alpha=0.12, label="IHDS-II survey window")

    # Single combined legend
    h1, l1 = ax1.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax1.legend(h1 + h2, l1 + l2, loc="upper left", framealpha=0.9)

    fig.tight_layout()
    out_fig = FIG_DIR / "14_cpi_india.png"
    fig.savefig(out_fig, dpi=200)
    plt.close(fig)
    print(f"Wrote {out_fig}")


# (script entry point; uncomment to re-run)
# main()


![14_cpi_india.png](figures/14_cpi_india.png)

_Figure produced by the cell above (saved to `figures/14_cpi_india.png`)._

## 6. Macro-Anchored Trajectory Simulator

For each household, builds a 24-month trajectory of (income, expense, savings, debt) from its IHDS-II annual anchor, modulated by CPI and damped by Poisson-distributed shocks (medical, disaster, crop-failure) at empirically-calibrated rates from the IHDS-II MI module.

In [ ]:
"""
Macro-anchored monthly household-trajectory simulator for the temporal early-warning
extension.

For each IHDS-II household h with annual snapshot (INCOME, COTOTAL, ASSETS, DB5+DB6A,
earner mix), the simulator produces a 24-month trajectory of (income, expense, savings,
debt) and a binary indicator of shocks each month. Trajectories are anchored to the
household's annual values and modulated by:

  (i)   the real India CPI-Combined monthly series (2011-01 .. 2012-12),
  (ii)  a household-specific income-volatility coefficient derived from earner mix,
  (iii) a household-specific consumption-smoothing coefficient,
  (iv)  Poisson-distributed shocks (medical, disaster, crop-failure) at rates
        calibrated to the IHDS-II 5-year-recall MI-module prevalence.

The mathematical model is:

  CPI scaling     m_t        = CPI_t / CPI_anchor

  Income          y_h(t)     = (Y_h / 12) * m_t * exp( eps_y^h(t) - sigma_y^h^2 / 2 )
                               * (1 - delta_crop * I[shock_crop(t)])

  Expense         x_h(t)     = (C_h / 12) * m_t * seasonal(t)
                               * exp( eps_x^h(t) - sigma_x^h^2 / 2 )
                               + (medical_spike + disaster_spike)

  Cash-flow gap   g_h(t)     = y_h(t) - x_h(t) - i * D_h(t-1)        (i = 2%/month service)
  Savings         S_h(t)     = max(0, S_h(t-1) + g_h(t))             if g_h(t) > 0,
                               max(0, S_h(t-1) + g_h(t))             else
  Debt            D_h(t)     = D_h(t-1) + max(0, -(S_h(t-1) + g_h(t))) (debt absorbs negative cash-flow
                               beyond exhausted savings)
                               - min(repay_h(t), D_h(t-1))           if g_h(t) > 0 and S_h(t-1) > 0

The shock variables eps_y, eps_x are i.i.d. N(0, sigma^2) and the shock indicators are
independent Bernoulli per month at calibrated rates.

Outputs:
  early_warning/simulated_panel.parquet         (~24 * 42152 rows; long format)
  early_warning/simulator_calibration.json      (empirical shock rates and CV used)
  figures/15_simulator_trajectories.png         (6 example household trajectories)
  figures/16_simulator_validation_marginals.png (simulated annual sums vs IHDS anchors)
  figures/17_simulator_shock_calibration.png    (empirical vs realised shock incidence)
"""
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path(__file__).resolve().parent.parent
HH_FILE = ROOT / "DS0002" / "36151-0002-Data.dta"
FFI_FILE = ROOT / "ihds2_ffi.parquet"
CPI_FILE = ROOT / "data" / "cpi_india.csv"
MANIFEST = ROOT / "early_warning" / "sim_variable_manifest.json"
OUT_PARQUET = ROOT / "early_warning" / "simulated_panel.parquet"
OUT_CAL_JSON = ROOT / "early_warning" / "simulator_calibration.json"
FIG_DIR = ROOT / "figures"
FIG_DIR.mkdir(exist_ok=True)

RNG_SEED = 42
N_MONTHS = 24
# Survey window centered at 2012-01 (mid-IHDS-II); month 1 = 2011-01, month 24 = 2012-12
CPI_ANCHOR_MONTH = "2012-01-01"
CPI_START_MONTH = "2011-01-01"

# Service rate on outstanding debt (monthly). Calibrated to ~25% APR on informal credit.
DEBT_MONTHLY_INTEREST = 0.02

# Initial liquid-savings buffer as a fraction of assets stock
INITIAL_SAVINGS_FRACTION = 0.05


def load_inputs() -> tuple[pd.DataFrame, pd.DataFrame, dict]:
    print("Loading manifest, household file, FFI labels, CPI...")
    with open(MANIFEST) as f:
        manifest = json.load(f)

    needed_cols = (
        ["STATEID", "DISTID", "PSUID", "HHID", "HHSPLITID"]
        + list(manifest["anchor_vars"].values())
        + manifest["earner_count_vars"]
        + manifest["composition_vars"]
        + [v["code"] for v in manifest["shock_vars"].values()]
        + manifest["debt_vars"]
    )
    needed_cols = list(dict.fromkeys(needed_cols))  # de-dup, preserve order

    hh = pd.read_stata(HH_FILE, columns=needed_cols, convert_categoricals=False)
    print(f"  household file: {hh.shape}")

    # IHDS missing codes -> NaN
    for code in (-9, -8, -7):
        hh = hh.replace(code, np.nan)

    # Re-build hh_id consistent with the cross-sectional pipeline
    hh_keys = ["STATEID", "DISTID", "PSUID", "HHID", "HHSPLITID"]
    hh["hh_id"] = hh[hh_keys].astype(str).agg("-".join, axis=1)

    cpi = pd.read_csv(CPI_FILE, parse_dates=["month"])
    return hh, cpi, manifest


def select_24mo_cpi_window(cpi: pd.DataFrame) -> tuple[np.ndarray, float]:
    """Slice the bundled CPI series to the 24 months centered on IHDS-II."""
    cpi = cpi.set_index("month")
    start = pd.Timestamp(CPI_START_MONTH)
    window = cpi.loc[start:start + pd.DateOffset(months=N_MONTHS - 1)]
    assert len(window) == N_MONTHS, f"need {N_MONTHS} months, got {len(window)}"
    cpi_anchor = float(cpi.loc[CPI_ANCHOR_MONTH, "cpi"])
    cpi_series = window["cpi"].to_numpy()
    return cpi_series, cpi_anchor


def calibrate_shock_rates(hh: pd.DataFrame, manifest: dict) -> dict:
    """Convert 5-year-recall MI* prevalence to monthly Poisson rates.

    MI1/MI2/MI5 in IHDS-II are coded as 1 if the household reports having
    suffered the shock in the past 5 years, 0 otherwise. Under a constant-hazard
    assumption, monthly rate = prevalence / 60.
    """
    rates = {}
    for name, spec in manifest["shock_vars"].items():
        col = spec["code"]
        prev = hh[col].mean(skipna=True)
        rate = float(prev) / 60.0
        rates[name] = {
            "code": col,
            "five_year_prevalence": float(prev),
            "monthly_poisson_rate": rate,
            "shock_kind": spec["shock_kind"],
        }
        print(f"  shock {name:14s} ({col}): 5y prev = {prev:.3f}, monthly rate = {rate:.5f}")
    return rates


def income_volatility(earner_counts: np.ndarray) -> np.ndarray:
    """Return per-household monthly log-income volatility sigma_y.

    earner_counts shape: (N, 5) for [NONAG, AGLAB, SALARY, BUSINESS, FARM]
    Heuristic: more earner diversification -> lower volatility.
    """
    total = earner_counts.sum(axis=1)
    # Herfindahl across earner buckets (1 = single source, 0 = diversified)
    with np.errstate(invalid="ignore", divide="ignore"):
        share = np.where(total[:, None] > 0, earner_counts / np.where(total[:, None] > 0, total[:, None], 1), 0)
    herf = (share ** 2).sum(axis=1)
    # Sole-earner farm / business / aglab households are highest volatility
    farm_or_aglab = earner_counts[:, [1, 4]].sum(axis=1) > 0   # AGLAB or FARM
    salary_primary = earner_counts[:, 2] >= total / 2          # majority-salary
    sigma = np.where(salary_primary, 0.10, 0.20)
    sigma = sigma + 0.10 * (farm_or_aglab & (total <= 1))      # +0.10 if sole farm/aglab
    sigma = sigma * (1 + 0.5 * (herf - 1.0 / 5.0))             # scale by concentration
    return np.clip(sigma, 0.05, 0.45)


def seasonal_factor(months_idx: np.ndarray) -> np.ndarray:
    """Seasonal expense multiplier indexed by month-of-year (1..12).

    Two festive bumps (Oct-Nov: +15%) and a small lean stretch (Mar-Apr: -5%).
    """
    m = months_idx
    s = np.ones_like(m, dtype=np.float32)
    s = np.where((m == 10) | (m == 11), 1.15, s)
    s = np.where((m == 3) | (m == 4), 0.95, s)
    return s.astype(np.float32)


def simulate(hh: pd.DataFrame, cpi: pd.DataFrame, manifest: dict) -> tuple[pd.DataFrame, dict]:
    rng = np.random.default_rng(RNG_SEED)
    cpi_series, cpi_anchor = select_24mo_cpi_window(cpi)
    m_t = cpi_series / cpi_anchor                              # (T,)
    print(f"  CPI window length: {len(m_t)}, anchor = {cpi_anchor}")

    # Calibrate Poisson rates from MI*
    rates = calibrate_shock_rates(hh, manifest)

    # Drop rows missing core anchors; impute zeros for the rest
    core = ["INCOME", "COTOTAL", "ASSETS", "DB5", "DB6A"]
    hh = hh.dropna(subset=["INCOME"]).reset_index(drop=True)
    for c in core:
        hh[c] = hh[c].fillna(0.0).astype(np.float64)

    N = len(hh)
    T = N_MONTHS
    print(f"  households simulated: {N:,}, T = {T} months")

    # Per-household constants
    Y = hh["INCOME"].to_numpy()                                # annual income
    C = hh["COTOTAL"].to_numpy().clip(min=1.0)                 # annual expense
    A = hh["ASSETS"].to_numpy().clip(min=0.0)                  # asset stock
    D0 = (hh["DB5"].fillna(0) + hh["DB6A"].fillna(0)).to_numpy()
    earner_counts = hh[manifest["earner_count_vars"]].fillna(0).to_numpy().astype(np.float32)
    sigma_y = income_volatility(earner_counts).astype(np.float32)
    sigma_x = np.full(N, 0.10, dtype=np.float32)               # consumption smoothing

    # Months-of-year for the 24-month window (Jan-2011 = month 1 ... Dec-2012 = month 12 wrap)
    start = pd.Timestamp(CPI_START_MONTH)
    moy = np.array([
        (start + pd.DateOffset(months=t)).month for t in range(T)
    ], dtype=np.int32)
    seasonal = seasonal_factor(moy)                            # (T,)

    # Base monthly anchors
    y_base = (Y[:, None] / 12.0) * m_t[None, :]                # (N, T)
    x_base = (C[:, None] / 12.0) * m_t[None, :] * seasonal[None, :]

    # Income / expense noise
    eps_y = rng.normal(0.0, sigma_y[:, None], size=(N, T)).astype(np.float32)
    eps_x = rng.normal(0.0, sigma_x[:, None], size=(N, T)).astype(np.float32)
    y = y_base * np.exp(eps_y - 0.5 * sigma_y[:, None] ** 2)
    x = x_base * np.exp(eps_x - 0.5 * sigma_x[:, None] ** 2)

    # ---- Shock injection ----
    # Medical: expense spike = lognormal(mu=log(0.5*income), sigma=0.8)
    p_med = rates["medical"]["monthly_poisson_rate"]
    p_dis = rates["disaster"]["monthly_poisson_rate"]
    p_crop = rates["crop_failure"]["monthly_poisson_rate"]

    # Restrict crop shock to households with farm earners (NWKFARM > 0)
    farm_mask = earner_counts[:, 4] > 0                        # (N,)

    sh_med = rng.binomial(1, p_med, size=(N, T)).astype(np.float32)
    sh_dis = rng.binomial(1, p_dis, size=(N, T)).astype(np.float32)
    sh_crop = rng.binomial(1, p_crop, size=(N, T)).astype(np.float32) * farm_mask[:, None]

    # Magnitudes (rupees), drawn from lognormals anchored on monthly income.
    # Clip Y to a small positive floor (~₹1000/yr) so log() is well-defined for
    # households with reported zero or negative annual income (rare, but present).
    Y_pos = np.clip(Y, 1000.0, None)
    med_amount  = np.exp(rng.normal(np.log(0.5 * (Y_pos[:, None] / 12.0)), 0.8, size=(N, T)))
    dis_amount  = np.exp(rng.normal(np.log(0.3 * (Y_pos[:, None] / 12.0)), 0.6, size=(N, T)))
    crop_drop   = rng.uniform(0.30, 0.70, size=(N, T)).astype(np.float32)  # multiplicative income drop

    # Apply shocks
    x = x + sh_med * med_amount + sh_dis * dis_amount
    y = y * (1.0 - sh_crop * crop_drop)

    # ---- Sequential debt / savings dynamics ----
    S = np.zeros((N, T), dtype=np.float32)
    D = np.zeros((N, T), dtype=np.float32)
    g = np.zeros((N, T), dtype=np.float32)
    S_prev = (INITIAL_SAVINGS_FRACTION * A).astype(np.float32)
    D_prev = D0.astype(np.float32)
    for t in range(T):
        service = DEBT_MONTHLY_INTEREST * D_prev
        gap_t = y[:, t] - x[:, t] - service
        g[:, t] = gap_t

        # Negative gap: pull from savings first; remainder pushed onto debt
        neg = -np.minimum(gap_t, 0.0)
        pull = np.minimum(neg, S_prev)
        roll_to_debt = neg - pull

        # Positive gap: top up savings; then repay debt up to 30% of monthly positive gap
        pos = np.maximum(gap_t, 0.0)
        repay = np.minimum(0.3 * pos, D_prev)
        save_add = pos - repay

        S_cur = np.clip(S_prev - pull + save_add, 0.0, None)
        D_cur = np.clip(D_prev + roll_to_debt - repay, 0.0, None)

        S[:, t] = S_cur
        D[:, t] = D_cur
        S_prev, D_prev = S_cur, D_cur

    # ---- Assemble long-format panel ----
    print("  assembling long-format DataFrame...")
    hh_id_arr = np.repeat(hh["hh_id"].to_numpy(), T)
    month_idx = np.tile(np.arange(1, T + 1), N)
    cpi_arr = np.tile(cpi_series, N)
    panel = pd.DataFrame({
        "hh_id":   hh_id_arr,
        "month":   month_idx,
        "income":  y.reshape(-1),
        "expense": x.reshape(-1),
        "gap":     g.reshape(-1),
        "savings": S.reshape(-1),
        "debt":    D.reshape(-1),
        "shock_medical":  sh_med.reshape(-1).astype(np.int8),
        "shock_disaster": sh_dis.reshape(-1).astype(np.int8),
        "shock_crop":     sh_crop.reshape(-1).astype(np.int8),
        "cpi":     cpi_arr,
    })

    print(f"  panel rows: {len(panel):,}")
    panel.to_parquet(OUT_PARQUET, index=False)
    print(f"Wrote {OUT_PARQUET}")

    calibration = {
        "n_households": int(N),
        "n_months": int(T),
        "cpi_window_start": CPI_START_MONTH,
        "cpi_anchor_month": CPI_ANCHOR_MONTH,
        "cpi_anchor_value": cpi_anchor,
        "debt_monthly_interest": DEBT_MONTHLY_INTEREST,
        "initial_savings_fraction": INITIAL_SAVINGS_FRACTION,
        "shock_rates": rates,
        "income_volatility_summary": {
            "mean": float(sigma_y.mean()),
            "p25":  float(np.percentile(sigma_y, 25)),
            "p50":  float(np.percentile(sigma_y, 50)),
            "p75":  float(np.percentile(sigma_y, 75)),
        },
    }
    with open(OUT_CAL_JSON, "w") as f:
        json.dump(calibration, f, indent=2)
    print(f"Wrote {OUT_CAL_JSON}")

    return panel, {"y": y, "x": x, "S": S, "D": D, "g": g,
                   "sh_med": sh_med, "sh_dis": sh_dis, "sh_crop": sh_crop,
                   "Y_anchor": Y, "C_anchor": C, "rates": rates, "hh": hh}


def make_validation_figures(arrays: dict, cpi_series: np.ndarray):
    print("Making validation figures...")
    y, x, S, D = arrays["y"], arrays["x"], arrays["S"], arrays["D"]
    Y_anchor, C_anchor = arrays["Y_anchor"], arrays["C_anchor"]
    sh_med, sh_dis, sh_crop = arrays["sh_med"], arrays["sh_dis"], arrays["sh_crop"]
    rates = arrays["rates"]
    plt.rcParams.update({"figure.dpi": 130, "savefig.dpi": 200, "font.size": 10})

    rng = np.random.default_rng(0)
    months = np.arange(1, y.shape[1] + 1)

    # ---- Fig 15: 6 example trajectories (3 stable, 3 stressed) ----
    # Pick stable = first 3 households with no shocks AND positive ending savings
    no_shock = (sh_med.sum(axis=1) == 0) & (sh_dis.sum(axis=1) == 0) & (sh_crop.sum(axis=1) == 0)
    stable_idx = np.where(no_shock & (S[:, -1] > 0))[0][:3]
    # Stressed = households whose debt grows by more than 50% over the window
    growth = (D[:, -1] - D[:, 0])
    stressed_idx = np.argsort(-growth)[:3]
    pick = np.concatenate([stable_idx, stressed_idx])
    labels = ["Stable A", "Stable B", "Stable C", "Stressed A", "Stressed B", "Stressed C"]

    fig, axes = plt.subplots(2, 3, figsize=(13, 7), sharex=True)
    for ax, idx, lbl in zip(axes.flat, pick, labels):
        ax.plot(months, y[idx] / 1000, label="Income", color="C0")
        ax.plot(months, x[idx] / 1000, label="Expense", color="C3")
        ax.plot(months, S[idx] / 1000, label="Savings", color="C2", ls="--")
        ax.plot(months, D[idx] / 1000, label="Debt", color="C1", ls=":")
        # Mark shocks
        sh_months = np.where(sh_med[idx] + sh_dis[idx] + sh_crop[idx] > 0)[0]
        for m in sh_months:
            ax.axvline(m + 1, color="purple", alpha=0.25, lw=2)
        ax.set_title(lbl, fontsize=10)
        ax.set_ylabel("₹ '000")
        ax.grid(alpha=0.3)
    axes[0, 0].legend(loc="upper left", fontsize=8)
    axes[-1, 1].set_xlabel("Month (1 = Jan 2011)")
    fig.suptitle("Simulated 24-month trajectories — three stable and three stressed households\n"
                 "Vertical purple lines mark shock months (medical / disaster / crop)", y=1.02)
    fig.tight_layout()
    fig.savefig(FIG_DIR / "15_simulator_trajectories.png", bbox_inches="tight")
    plt.close(fig)

    # ---- Fig 16: annual sums vs IHDS anchors (validation scatter) ----
    annual_income_sim = y.sum(axis=1)
    annual_expense_sim = x.sum(axis=1)
    sample = rng.choice(len(Y_anchor), size=min(2000, len(Y_anchor)), replace=False)

    fig, axes = plt.subplots(1, 2, figsize=(11, 5))
    ax = axes[0]
    ax.scatter(Y_anchor[sample] / 1e5, annual_income_sim[sample] / 1e5, s=4, alpha=0.4)
    lim = max(Y_anchor[sample].max(), annual_income_sim[sample].max()) / 1e5 * 1.05
    ax.plot([0, lim], [0, lim], "r--", lw=1)
    ax.set_xlabel("IHDS-II annual INCOME (₹ lakh)")
    ax.set_ylabel("Simulated annual sum (₹ lakh)")
    ax.set_title("Validation: simulated annual income matches IHDS anchor")
    ax.set_xlim(0, lim); ax.set_ylim(0, lim); ax.grid(alpha=0.3)

    ax = axes[1]
    ax.scatter(C_anchor[sample] / 1e5, annual_expense_sim[sample] / 1e5, s=4, alpha=0.4, color="C3")
    lim2 = max(C_anchor[sample].max(), annual_expense_sim[sample].max()) / 1e5 * 1.05
    ax.plot([0, lim2], [0, lim2], "r--", lw=1)
    ax.set_xlabel("IHDS-II annual COTOTAL (₹ lakh)")
    ax.set_ylabel("Simulated annual sum (₹ lakh)")
    ax.set_title("Validation: simulated annual expense matches IHDS anchor")
    ax.set_xlim(0, lim2); ax.set_ylim(0, lim2); ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(FIG_DIR / "16_simulator_validation_marginals.png", bbox_inches="tight")
    plt.close(fig)

    # ---- Fig 17: realised shock incidence vs target Poisson rate ----
    fig, ax = plt.subplots(figsize=(7, 4))
    names = ["medical", "disaster", "crop_failure"]
    targets = [rates[n]["monthly_poisson_rate"] for n in names]
    realised = [sh_med.mean(), sh_dis.mean(), sh_crop.mean()]
    width = 0.35
    xs = np.arange(len(names))
    ax.bar(xs - width / 2, targets, width, label="Target Poisson rate (prev/60)")
    ax.bar(xs + width / 2, realised, width, label="Realised monthly incidence")
    ax.set_xticks(xs); ax.set_xticklabels(names)
    ax.set_ylabel("Monthly probability")
    ax.set_title("Shock-rate calibration: target vs realised")
    ax.legend(); ax.grid(alpha=0.3, axis="y")
    fig.tight_layout()
    fig.savefig(FIG_DIR / "17_simulator_shock_calibration.png", bbox_inches="tight")
    plt.close(fig)

    print("  wrote figures/15, 16, 17")


def main():
    hh, cpi, manifest = load_inputs()
    panel, arrays = simulate(hh, cpi, manifest)
    cpi_series, _ = select_24mo_cpi_window(cpi)
    make_validation_figures(arrays, cpi_series)
    print("\nDone.")


# (script entry point; uncomment to re-run)
# main()


![15_simulator_trajectories.png](figures/15_simulator_trajectories.png)

_Figure produced by the cell above (saved to `figures/15_simulator_trajectories.png`)._

## 7. Time-Varying FFI

Implements the project plan's flow-based formula
$\text{FFI}_t = w_1 \cdot \text{DebtRatio}_t + w_2 \cdot \text{ExpenseVolatility}_t - w_3 \cdot \text{SavingsBuffer}_t$
with equal weights, defines the supervised 3-month-ahead stress event
$y_t = \mathbb{I}\{\max(\text{FFI}_{t+1}, \text{FFI}_{t+2}, \text{FFI}_{t+3}) > \tau\}$
where $\tau$ is the 75th percentile of FFI$_t$ (matching the cross-sectional Fragile cutoff).

In [ ]:
"""
Compute the time-varying Financial Fragility Index FFI_t on the simulated panel,
per the project plan:

    FFI_t = w1 * DebtRatio_t  +  w2 * ExpenseVolatility_t  -  w3 * SavingsBuffer_t

with:
    DebtRatio_t          = debt_t / max(monthly_income_t, eps)
    ExpenseVolatility_t  = rolling 6-month std(expense) / rolling 6-month mean(expense)
    SavingsBuffer_t      = savings_t / max(monthly_expense_t, eps)        (months of cover)

We standardize each component (z-score across the entire panel pool) and use
equal weights w1 = w2 = w3 = 1/3, mirroring the cross-sectional FFI from §2 of
the report. Higher FFI_t = more fragile.

A "stress event" within a 3-month look-ahead window is defined as

    y_t = I[ max(FFI_{t+1}, FFI_{t+2}, FFI_{t+3}) > tau ]

with the threshold tau set to the cross-sectional 75th percentile of FFI_t
(matching the "Fragile or Distressed" quartile split used in the cross-sectional
work). This means the binary problem is to predict whether a household will
enter the top quartile of fragility within the next quarter, given its
preceding history.

Outputs:
  early_warning/panel_with_ffi.parquet          (hh × month × features + ffi_t + components)
  early_warning/panel_supervised.parquet        (one row per (hh, t) with all features and y_t)
  early_warning/ffi_calibration.json            (weights, threshold tau, summary stats)
  figures/18_ffi_t_distribution.png             (marginal of FFI_t and tau)
  figures/19_ffi_t_components_corr.png          (component correlation heatmap)
  figures/20_ffi_t_sample_paths.png             (FFI_t paths for 6 illustrative households)
"""
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path(__file__).resolve().parent.parent
IN_PARQUET = ROOT / "early_warning" / "simulated_panel.parquet"
OUT_PANEL = ROOT / "early_warning" / "panel_with_ffi.parquet"
OUT_SUPERVISED = ROOT / "early_warning" / "panel_supervised.parquet"
OUT_CAL = ROOT / "early_warning" / "ffi_calibration.json"
FIG_DIR = ROOT / "figures"

ROLLING_WINDOW = 6        # months for expense-volatility window
HORIZON = 3               # months ahead for stress-event lookahead
W1, W2, W3 = 1/3, 1/3, 1/3
EPS = 1.0                 # divide-by-zero guard, in rupees


def load_panel() -> pd.DataFrame:
    print(f"Loading {IN_PARQUET}...")
    df = pd.read_parquet(IN_PARQUET)
    print(f"  rows: {len(df):,}  households: {df['hh_id'].nunique():,}  months: {df['month'].nunique()}")
    return df.sort_values(["hh_id", "month"]).reset_index(drop=True)


def compute_components(df: pd.DataFrame) -> pd.DataFrame:
    """Add DebtRatio_t, ExpenseVolatility_t, SavingsBuffer_t columns (raw, not z-scored)."""
    print("Computing per-month FFI components (raw)...")
    df = df.copy()
    df["debt_ratio_raw"] = df["debt"] / (df["income"].abs() + EPS)
    df["savings_buffer_raw"] = df["savings"] / (df["expense"].abs() + EPS)

    # Rolling 6-month expense volatility (coefficient of variation), per household
    g = df.groupby("hh_id", sort=False)["expense"]
    roll_mean = g.transform(lambda s: s.rolling(ROLLING_WINDOW, min_periods=2).mean())
    roll_std  = g.transform(lambda s: s.rolling(ROLLING_WINDOW, min_periods=2).std(ddof=0))
    df["expense_vol_raw"] = roll_std / (roll_mean.abs() + EPS)

    # First month with no rolling window: forward-fill from t=2 onwards within hh
    # (rolling at t=1 is undefined; we mask later when building supervised set)
    return df


def standardize_components(df: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    """Z-score the three components across the entire pool (households × months),
    using only months where all three are finite. Return df with z-cols and the
    standardization statistics for inversion / reporting."""
    print("Standardizing components...")
    comps = ["debt_ratio_raw", "expense_vol_raw", "savings_buffer_raw"]
    # Winsorize at the 99th percentile to tame heavy tails before z-scoring
    stats = {}
    for c in comps:
        finite = df[c].replace([np.inf, -np.inf], np.nan).dropna()
        p1 = float(finite.quantile(0.01))
        p99 = float(finite.quantile(0.99))
        df[c + "_w"] = df[c].clip(lower=p1, upper=p99)
        finite_w = df[c + "_w"].replace([np.inf, -np.inf], np.nan).dropna()
        mu = float(finite_w.mean())
        sd = float(finite_w.std(ddof=0))
        stats[c] = {"p1": p1, "p99": p99, "mean": mu, "std": sd}
        df[c.replace("_raw", "_z")] = (df[c + "_w"] - mu) / (sd if sd > 0 else 1.0)
        df = df.drop(columns=[c + "_w"])
    return df, stats


def aggregate_ffi(df: pd.DataFrame) -> pd.DataFrame:
    """Equal-weighted aggregation per the plan formula."""
    print("Aggregating FFI_t = w1*DebtRatio + w2*ExpenseVol - w3*SavingsBuffer...")
    df["ffi_t"] = (
        W1 * df["debt_ratio_z"]
      + W2 * df["expense_vol_z"]
      - W3 * df["savings_buffer_z"]
    )
    return df


def build_supervised(df: pd.DataFrame, horizon: int = HORIZON,
                     warmup: int = ROLLING_WINDOW) -> tuple[pd.DataFrame, float]:
    """For each (hh, t), compute the binary stress event:

        y_t = 1 if max(ffi_{t+1}, ..., ffi_{t+horizon}) > tau, else 0

    tau is set to the 75th percentile of ffi_t over all rows where the rolling
    window is fully populated. Returns the supervised long-format frame and tau.
    """
    print(f"Building supervised target (horizon={horizon}, warmup={warmup})...")
    df = df.copy()

    # Threshold tau (top-quartile of FFI_t after warmup, matching cross-sectional cut)
    valid_for_tau = df[df["month"] > warmup]["ffi_t"]
    tau = float(valid_for_tau.quantile(0.75))
    print(f"  tau = 75th percentile of FFI_t (after warmup) = {tau:.4f}")

    # Build max future FFI within (t+1, t+horizon) per household
    g = df.groupby("hh_id", sort=False)["ffi_t"]
    fwd_max = g.transform(lambda s: s.shift(-1).rolling(horizon, min_periods=1).max())
    df["fwd_max_ffi"] = fwd_max
    df["y_stress"] = (df["fwd_max_ffi"] > tau).astype("Int8")

    # Drop rows where look-ahead is incomplete (last `horizon` months per hh)
    # and where rolling-vol warmup is not yet done.
    df["valid"] = (
        df["month"].between(warmup + 1, df["month"].max() - horizon).astype(int)
    )
    sup = df[df["valid"] == 1].copy()
    print(f"  supervised rows: {len(sup):,}  positive rate: {sup['y_stress'].mean():.3f}")
    return sup, tau


def make_figures(df: pd.DataFrame, sup: pd.DataFrame, tau: float, stats: dict):
    print("Making figures 18–20...")
    plt.rcParams.update({"figure.dpi": 130, "savefig.dpi": 200, "font.size": 10})
    rng = np.random.default_rng(0)

    # --- Fig 18: Distribution of FFI_t and the tau line ---
    fig, ax = plt.subplots(figsize=(7, 4))
    sample = df.loc[df["month"] > ROLLING_WINDOW, "ffi_t"].sample(
        n=min(200_000, len(df)), random_state=0
    )
    ax.hist(sample, bins=120, color="C0", alpha=0.85)
    ax.axvline(tau, color="C3", ls="--", lw=2,
               label=f"$\\tau$ = 75th pct = {tau:.2f}")
    ax.set_xlabel("FFI$_t$ (time-varying FFI, equal-weight z-score)")
    ax.set_ylabel("Number of (household, month) cells")
    ax.set_title("Distribution of the time-varying FFI$_t$ across simulated panel")
    ax.legend(); ax.grid(alpha=0.3)
    fig.tight_layout(); fig.savefig(FIG_DIR / "18_ffi_t_distribution.png"); plt.close(fig)

    # --- Fig 19: Component correlation heatmap (after standardisation) ---
    corr = sup[["debt_ratio_z", "expense_vol_z", "savings_buffer_z", "ffi_t"]].corr()
    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(corr.values, cmap="coolwarm", vmin=-1, vmax=1)
    ax.set_xticks(range(4)); ax.set_xticklabels(corr.columns, rotation=30, ha="right")
    ax.set_yticks(range(4)); ax.set_yticklabels(corr.columns)
    for (i, j), v in np.ndenumerate(corr.values):
        ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                color="white" if abs(v) > 0.5 else "black", fontsize=9)
    fig.colorbar(im, ax=ax, fraction=0.046)
    ax.set_title("Pearson correlation of FFI$_t$ components and aggregate")
    fig.tight_layout(); fig.savefig(FIG_DIR / "19_ffi_t_components_corr.png"); plt.close(fig)

    # --- Fig 20: Six sample household FFI_t paths (3 always-stable, 3 crossing tau) ---
    grouped = df.groupby("hh_id")
    max_ffi = grouped["ffi_t"].max()
    min_ffi = grouped["ffi_t"].min()

    always_stable = max_ffi[(max_ffi < tau - 0.1)].index.tolist()[:3]
    transitions = grouped["ffi_t"].apply(
        lambda s: ((s.shift(1) <= tau) & (s > tau)).sum()
    )
    crossers = transitions[transitions >= 1].index.tolist()[:3]

    pick = list(always_stable) + list(crossers)
    titles = ["Always stable A", "Always stable B", "Always stable C",
              "Crosses τ — case A", "Crosses τ — case B", "Crosses τ — case C"]

    fig, axes = plt.subplots(2, 3, figsize=(13, 6), sharex=True)
    for ax, hh, ttl in zip(axes.flat, pick, titles):
        series = df.loc[df["hh_id"] == hh, ["month", "ffi_t"]].sort_values("month")
        ax.plot(series["month"], series["ffi_t"], color="C0", lw=1.5)
        ax.axhline(tau, color="C3", ls="--", lw=1, label=f"τ = {tau:.2f}")
        ax.set_title(ttl, fontsize=10)
        ax.set_ylabel("FFI$_t$"); ax.grid(alpha=0.3)
    axes[0, 0].legend(loc="upper left", fontsize=8)
    axes[-1, 1].set_xlabel("Month (1 = Jan 2011)")
    fig.suptitle("Sample FFI$_t$ trajectories — three stable, three crossing the threshold", y=1.02)
    fig.tight_layout(); fig.savefig(FIG_DIR / "20_ffi_t_sample_paths.png", bbox_inches="tight")
    plt.close(fig)


def main():
    df = load_panel()
    df = compute_components(df)
    df, stats = standardize_components(df)
    df = aggregate_ffi(df)
    sup, tau = build_supervised(df, horizon=HORIZON, warmup=ROLLING_WINDOW)

    # Persist the (hh, t) panel with FFI for the LSTM input pipeline
    keep_panel = [
        "hh_id", "month", "income", "expense", "savings", "debt",
        "shock_medical", "shock_disaster", "shock_crop", "cpi",
        "debt_ratio_raw", "expense_vol_raw", "savings_buffer_raw",
        "debt_ratio_z", "expense_vol_z", "savings_buffer_z",
        "ffi_t",
    ]
    df[keep_panel].to_parquet(OUT_PANEL, index=False)
    print(f"Wrote {OUT_PANEL}  ({len(df):,} rows)")

    keep_sup = keep_panel + ["fwd_max_ffi", "y_stress"]
    sup[keep_sup].to_parquet(OUT_SUPERVISED, index=False)
    print(f"Wrote {OUT_SUPERVISED}  ({len(sup):,} rows)")

    with open(OUT_CAL, "w") as f:
        json.dump({
            "weights": {"w1_debt_ratio": W1, "w2_expense_vol": W2, "w3_savings_buffer": W3},
            "rolling_window_months": ROLLING_WINDOW,
            "horizon_months": HORIZON,
            "tau_75th_pct": tau,
            "component_stats": stats,
            "positive_rate": float(sup["y_stress"].mean()),
            "n_supervised_rows": int(len(sup)),
        }, f, indent=2)
    print(f"Wrote {OUT_CAL}")

    make_figures(df, sup, tau, stats)
    print("\nDone.")


# (script entry point; uncomment to re-run)
# main()


![18_ffi_t_distribution.png](figures/18_ffi_t_distribution.png)

_Figure produced by the cell above (saved to `figures/18_ffi_t_distribution.png`)._

## 8. Building LSTM Training Tensors

Splits households 70/15/15 train/val/test, then for each (household, prediction-time) pair extracts a 12-month input sequence and the 3-month-ahead targets.

In [ ]:
"""
Sequence construction and train/val/test split for the LSTM early-warning model.

For each household h, the time-varying panel `panel_with_ffi.parquet` carries
24 months. For each prediction point t in [HISTORY_LEN, T - HORIZON], we build:

    X_seq[h, t]   = shape (HISTORY_LEN, n_time_features)   monthly time-series
                    spanning months (t - HISTORY_LEN + 1) .. t.
    X_static[h]   = shape (n_static_features,)             cross-sectional features.
    y_ffi[h, t]   = ffi_t at month t + HORIZON (point regression target).
    y_stress[h,t] = supervised binary stress event from `panel_supervised.parquet`.

Households are split 70/15/15 into train / val / test by hh_id (no household
appears in more than one split, preventing temporal leakage between train and
test sets).

The time features are z-scored using statistics computed on the training split
only. Static features are one-hot encoded using the cross-sectional pipeline's
column list. Outputs are pickled tensors ready for the LSTM training loop.

Outputs:
  early_warning/lstm_arrays/{train,val,test}_X_seq.npy
  early_warning/lstm_arrays/{train,val,test}_X_static.npy
  early_warning/lstm_arrays/{train,val,test}_y_ffi.npy
  early_warning/lstm_arrays/{train,val,test}_y_stress.npy
  early_warning/lstm_arrays/{train,val,test}_hh_id.npy
  early_warning/lstm_arrays/feature_spec.json
"""
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path(__file__).resolve().parent.parent
PANEL = ROOT / "early_warning" / "panel_with_ffi.parquet"
SUP = ROOT / "early_warning" / "panel_supervised.parquet"
HH_FILE = ROOT / "DS0002" / "36151-0002-Data.dta"
OUT_DIR = ROOT / "early_warning" / "lstm_arrays"
OUT_DIR.mkdir(exist_ok=True)

HISTORY_LEN = 12
HORIZON = 3
TOTAL_MONTHS = 24
RNG_SEED = 42

TIME_FEATURES = [
    "log_income", "log_expense", "log_savings", "log_debt",
    "debt_ratio_z", "expense_vol_z", "savings_buffer_z",
    "ffi_t", "cpi_n",
    "shock_medical", "shock_disaster", "shock_crop",
]

STATIC_COLS = [
    "MHEADAGE", "FHEADAGE",
    "HHEDUC", "HHEDUCM", "HHEDUCF",
    "ID11", "ID13", "GROUPS",
    "URBAN2011", "URBAN4_2011", "METRO", "METRO6",
    "HQ1", "HQWALL", "HQROOF", "HQFLOOR",
    "WATER", "SATOILET", "SAKITCHEN",
    "FU1", "FULPG",
    "MG1",
]
CATEGORICAL = [
    "ID11", "ID13", "GROUPS",
    "URBAN2011", "URBAN4_2011", "METRO", "METRO6",
    "HQWALL", "HQROOF", "HQFLOOR",
    "WATER", "SATOILET", "SAKITCHEN", "FU1", "FULPG", "MG1",
]


def load_static() -> pd.DataFrame:
    print("Loading static features from DS0002...")
    cols = ["STATEID", "DISTID", "PSUID", "HHID", "HHSPLITID"] + STATIC_COLS
    df = pd.read_stata(HH_FILE, columns=cols, convert_categoricals=False)
    for code in (-9, -8, -7):
        df = df.replace(code, np.nan)
    keys = ["STATEID", "DISTID", "PSUID", "HHID", "HHSPLITID"]
    df["hh_id"] = df[keys].astype(str).agg("-".join, axis=1)
    return df.drop(columns=keys)


def prepare_time_features(panel: pd.DataFrame) -> pd.DataFrame:
    print("Preparing time-feature columns...")
    panel = panel.copy()
    panel["log_income"]  = np.log1p(panel["income"].clip(lower=0))
    panel["log_expense"] = np.log1p(panel["expense"].clip(lower=0))
    panel["log_savings"] = np.log1p(panel["savings"].clip(lower=0))
    panel["log_debt"]    = np.log1p(panel["debt"].clip(lower=0))
    panel["cpi_n"]       = (panel["cpi"] - panel["cpi"].mean()) / panel["cpi"].std()
    # The first month per household has NaN expense_vol_z / savings_buffer_z /
    # debt_ratio_z / ffi_t because the 6-month rolling-window stat is undefined
    # at t=1. Z-scored features default cleanly to 0 (the pool mean).
    for col in ["expense_vol_z", "savings_buffer_z", "debt_ratio_z", "ffi_t"]:
        if col in panel.columns:
            panel[col] = panel[col].fillna(0.0)
    return panel


def fit_scalers(train_df: pd.DataFrame) -> dict:
    """Compute mean/std per time-feature on the training rows only."""
    stats = {}
    for c in TIME_FEATURES:
        if c.startswith("shock_"):
            stats[c] = {"mean": 0.0, "std": 1.0}
            continue
        mu = float(train_df[c].mean()); sd = float(train_df[c].std())
        if sd < 1e-8:
            sd = 1.0
        stats[c] = {"mean": mu, "std": sd}
    return stats


def apply_scalers(df: pd.DataFrame, stats: dict) -> pd.DataFrame:
    df = df.copy()
    for c, s in stats.items():
        df[c] = (df[c] - s["mean"]) / s["std"]
    return df


def build_sequences(panel: pd.DataFrame, sup: pd.DataFrame,
                    static_oh: pd.DataFrame, hh_ids: np.ndarray):
    """Build (X_seq, X_static, y_ffi, y_stress, hh_id_per_row) for households in `hh_ids`."""
    print(f"  building sequences for {len(hh_ids):,} households...")
    panel_sub = panel[panel["hh_id"].isin(set(hh_ids))]
    panel_idx = panel_sub.set_index(["hh_id", "month"]).sort_index()
    panel_arr = panel_idx[TIME_FEATURES].to_numpy(dtype=np.float32)
    # Index lookup
    hh_to_offset = {}
    for hh, sub in panel_idx.groupby(level=0, sort=False):
        hh_to_offset[hh] = sub.index.get_level_values("month").to_numpy()

    sup_sub = sup[sup["hh_id"].isin(set(hh_ids))]
    sup_sub = sup_sub.merge(
        panel_idx[["ffi_t"]].reset_index().rename(columns={"month": "month_t"}),
        left_on=["hh_id"], right_on=["hh_id"], how="left", suffixes=("", "_lookup"),
    )
    # Skip rows where the requested history window is incomplete
    sup_sub = sup_sub[(sup_sub["month"] >= HISTORY_LEN)
                    & (sup_sub["month"] <= TOTAL_MONTHS - HORIZON)]
    sup_sub = sup_sub[["hh_id", "month", "y_stress"]].drop_duplicates().reset_index(drop=True)

    # Build arrays
    N = len(sup_sub)
    X_seq = np.zeros((N, HISTORY_LEN, len(TIME_FEATURES)), dtype=np.float32)
    y_ffi = np.zeros(N, dtype=np.float32)
    y_stress = np.zeros(N, dtype=np.int8)
    hh_arr = np.empty(N, dtype=object)

    # panel reshape: (n_hh, T, n_feat)
    grouped = panel_sub.sort_values(["hh_id", "month"]).set_index(["hh_id", "month"])
    hh_list = sorted(set(panel_sub["hh_id"]))
    hh_to_idx = {hh: i for i, hh in enumerate(hh_list)}
    grid = (
        grouped.reindex(
            pd.MultiIndex.from_product([hh_list, range(1, TOTAL_MONTHS + 1)],
                                       names=["hh_id", "month"])
        )[TIME_FEATURES]
        .to_numpy(dtype=np.float32)
        .reshape(len(hh_list), TOTAL_MONTHS, len(TIME_FEATURES))
    )
    ffi_grid = (
        grouped.reindex(
            pd.MultiIndex.from_product([hh_list, range(1, TOTAL_MONTHS + 1)],
                                       names=["hh_id", "month"])
        )["ffi_t"]
        .to_numpy(dtype=np.float32)
        .reshape(len(hh_list), TOTAL_MONTHS)
    )

    for i, row in sup_sub.iterrows():
        hh = row["hh_id"]; t = int(row["month"])
        idx = hh_to_idx[hh]
        # History window: months [t - HISTORY_LEN + 1, t] (inclusive)
        start = t - HISTORY_LEN
        X_seq[i] = grid[idx, start:t, :]      # rows for months start+1 .. t in 1-indexed
        # ffi target at month t + HORIZON
        y_ffi[i] = ffi_grid[idx, t + HORIZON - 1]
        y_stress[i] = int(row["y_stress"])
        hh_arr[i] = hh

    # Static features (broadcast across rows)
    static_lookup = static_oh.set_index("hh_id")
    static_mat = static_lookup.loc[hh_arr].to_numpy(dtype=np.float32)
    return X_seq, static_mat, y_ffi, y_stress, hh_arr


def main():
    print("Loading panels...")
    panel = pd.read_parquet(PANEL)
    sup = pd.read_parquet(SUP)
    panel = prepare_time_features(panel)
    print(f"  panel rows: {len(panel):,}  supervised rows: {len(sup):,}")

    static = load_static()
    # One-hot encode categorical static features
    static_oh = pd.get_dummies(
        static, columns=[c for c in CATEGORICAL if c in static.columns],
        dummy_na=True, drop_first=True,
    )
    # Impute remaining numeric NaNs with column median
    for c in static_oh.columns:
        if c == "hh_id":
            continue
        if static_oh[c].isna().any():
            static_oh[c] = static_oh[c].fillna(static_oh[c].median())
    static_oh = static_oh.astype({c: np.float32 for c in static_oh.columns if c != "hh_id"})
    print(f"  static after one-hot: {static_oh.shape}")

    # Train/val/test split by household
    rng = np.random.default_rng(RNG_SEED)
    all_hh = panel["hh_id"].unique()
    rng.shuffle(all_hh)
    n = len(all_hh)
    n_tr = int(0.70 * n); n_va = int(0.15 * n)
    hh_train = all_hh[:n_tr]; hh_val = all_hh[n_tr:n_tr + n_va]; hh_test = all_hh[n_tr + n_va:]
    print(f"  split: train={len(hh_train):,} val={len(hh_val):,} test={len(hh_test):,}")

    # Fit scalers on training panel only
    print("Fitting scalers on training rows...")
    train_panel = panel[panel["hh_id"].isin(set(hh_train))]
    stats = fit_scalers(train_panel)
    panel_s = apply_scalers(panel, stats)

    # Build sequences for each split
    def _clean(X_seq, X_st, y_ffi, y_s, hh, name):
        # Targets must be finite; X is fillna'd to zero in-place (z-scored features
        # default cleanly to the pool mean of 0).
        bad = np.isnan(y_ffi) | np.isinf(y_ffi)
        n_bad = int(bad.sum())
        if n_bad:
            print(f"  {name}: dropping {n_bad:,} rows with NaN target "
                  f"({n_bad / len(y_ffi):.2%} of {len(y_ffi):,})")
        keep = ~bad
        X_seq = np.nan_to_num(X_seq[keep], nan=0.0, posinf=0.0, neginf=0.0)
        X_st  = np.nan_to_num(X_st[keep],  nan=0.0, posinf=0.0, neginf=0.0)
        return X_seq, X_st, y_ffi[keep], y_s[keep], hh[keep]

    print("Building train sequences...")
    Xtr, Str, ytr_ffi, ytr_stress, hh_tr = build_sequences(panel_s, sup, static_oh, hh_train)
    Xtr, Str, ytr_ffi, ytr_stress, hh_tr = _clean(Xtr, Str, ytr_ffi, ytr_stress, hh_tr, "train")
    print(f"  train: X_seq {Xtr.shape}  static {Str.shape}  pos rate {ytr_stress.mean():.3f}")
    print("Building val sequences...")
    Xva, Sva, yva_ffi, yva_stress, hh_va = build_sequences(panel_s, sup, static_oh, hh_val)
    Xva, Sva, yva_ffi, yva_stress, hh_va = _clean(Xva, Sva, yva_ffi, yva_stress, hh_va, "val")
    print(f"  val:   X_seq {Xva.shape}  static {Sva.shape}  pos rate {yva_stress.mean():.3f}")
    print("Building test sequences...")
    Xte, Ste, yte_ffi, yte_stress, hh_te = build_sequences(panel_s, sup, static_oh, hh_test)
    Xte, Ste, yte_ffi, yte_stress, hh_te = _clean(Xte, Ste, yte_ffi, yte_stress, hh_te, "test")
    print(f"  test:  X_seq {Xte.shape}  static {Ste.shape}  pos rate {yte_stress.mean():.3f}")

    np.save(OUT_DIR / "train_X_seq.npy", Xtr); np.save(OUT_DIR / "train_X_static.npy", Str)
    np.save(OUT_DIR / "train_y_ffi.npy", ytr_ffi); np.save(OUT_DIR / "train_y_stress.npy", ytr_stress)
    np.save(OUT_DIR / "train_hh_id.npy", hh_tr)
    np.save(OUT_DIR / "val_X_seq.npy", Xva); np.save(OUT_DIR / "val_X_static.npy", Sva)
    np.save(OUT_DIR / "val_y_ffi.npy", yva_ffi); np.save(OUT_DIR / "val_y_stress.npy", yva_stress)
    np.save(OUT_DIR / "val_hh_id.npy", hh_va)
    np.save(OUT_DIR / "test_X_seq.npy", Xte); np.save(OUT_DIR / "test_X_static.npy", Ste)
    np.save(OUT_DIR / "test_y_ffi.npy", yte_ffi); np.save(OUT_DIR / "test_y_stress.npy", yte_stress)
    np.save(OUT_DIR / "test_hh_id.npy", hh_te)

    spec = {
        "history_len": HISTORY_LEN, "horizon": HORIZON,
        "time_features": TIME_FEATURES,
        "n_time_features": len(TIME_FEATURES),
        "static_features": [c for c in static_oh.columns if c != "hh_id"],
        "n_static_features": int(Str.shape[1]),
        "scalers": stats,
        "split": {
            "n_train_hh": int(len(hh_train)),
            "n_val_hh": int(len(hh_val)),
            "n_test_hh": int(len(hh_test)),
            "n_train_rows": int(len(ytr_stress)),
            "n_val_rows":   int(len(yva_stress)),
            "n_test_rows":  int(len(yte_stress)),
        },
    }
    with open(OUT_DIR / "feature_spec.json", "w") as f:
        json.dump(spec, f, indent=2, default=str)
    print(f"Wrote {OUT_DIR}/")


# (script entry point; uncomment to re-run)
# main()


## 9. LSTM with MC Dropout — Training

Two-layer LSTM (hidden = 64) with inter-layer dropout (p = 0.2), joint regression + classification heads, Adam optimiser, early stopping on validation ROC-AUC. Runs on the Apple-Silicon MPS device.

In [ ]:
"""
LSTM with MC Dropout for the joint regression / binary-classification of
financial fragility 3 months ahead.

Architecture:
    sequence input  (B, L=12, F_t=12)
       -> LSTM(F_t, hidden=64, num_layers=2, dropout=0.2)
       -> take last hidden state h_L  (B, 64)
       -> concat with static features  (B, 64 + F_s)
       -> MLP  Linear(124, 64) -> ReLU -> Dropout(0.2) -> Linear(64, 2)
       -> outputs [ffi_pred, stress_logit]

Loss: 0.5 * MSE(ffi_pred, y_ffi)  +  BCEWithLogits(stress_logit, y_stress, pos_weight)

MC Dropout (Gal & Ghahramani 2016) makes the network a Bayesian approximation
by keeping dropout active at inference and treating K stochastic forward passes
as draws from the approximate posterior. We expose the dropout layers and the
model is set to .train() mode for inference sampling, with gradients disabled.

Outputs:
    early_warning/lstm_arrays/lstm_checkpoint.pt   (best-by-val-AUC weights)
    early_warning/lstm_arrays/lstm_train_log.json  (per-epoch losses and metrics)
    figures/21_lstm_training_curves.png            (train/val loss + val AUC vs epoch)
"""
from __future__ import annotations

import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import average_precision_score, roc_auc_score
from torch.utils.data import DataLoader, TensorDataset

ROOT = Path(__file__).resolve().parent.parent
ARR = ROOT / "early_warning" / "lstm_arrays"
FIG = ROOT / "figures"
CKPT = ARR / "lstm_checkpoint.pt"
LOG = ARR / "lstm_train_log.json"

EPOCHS = 30
BATCH = 512
LR = 1e-3
WD = 1e-5
PATIENCE = 5
SEED = 42

# --- Hyperparameters of the architecture ---
HIDDEN = 64
LSTM_LAYERS = 2
DROPOUT = 0.2
LOSS_WEIGHT_REG = 0.5         # weight on MSE term


def pick_device() -> torch.device:
    if torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


class LSTMEarlyWarning(nn.Module):
    def __init__(self, n_time_features: int, n_static_features: int,
                 hidden: int = HIDDEN, num_layers: int = LSTM_LAYERS,
                 dropout: float = DROPOUT):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=n_time_features,
            hidden_size=hidden,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        # MLP head on [last hidden state || static features]
        in_dim = hidden + n_static_features
        self.head = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 2),                # [ffi_pred, stress_logit]
        )

    def forward(self, x_seq: torch.Tensor, x_static: torch.Tensor) -> torch.Tensor:
        _, (h_n, _) = self.lstm(x_seq)
        h_last = h_n[-1]                          # (B, hidden), top-layer last hidden state
        z = torch.cat([h_last, x_static], dim=1)
        return self.head(z)                       # (B, 2)


def load_arrays():
    def _load(split):
        return (np.load(ARR / f"{split}_X_seq.npy"),
                np.load(ARR / f"{split}_X_static.npy"),
                np.load(ARR / f"{split}_y_ffi.npy"),
                np.load(ARR / f"{split}_y_stress.npy"))
    return _load("train"), _load("val"), _load("test")


def make_loader(X_seq, X_st, y_ffi, y_s, batch, shuffle):
    ds = TensorDataset(
        torch.from_numpy(X_seq), torch.from_numpy(X_st),
        torch.from_numpy(y_ffi), torch.from_numpy(y_s.astype(np.float32)),
    )
    return DataLoader(ds, batch_size=batch, shuffle=shuffle, num_workers=0, drop_last=False)


def evaluate(model, loader, device, pos_weight):
    model.eval()
    losses = []
    probs, ys, ffis_pred, ffis_true = [], [], [], []
    with torch.no_grad():
        for X_seq, X_st, y_ffi, y_s in loader:
            X_seq = X_seq.to(device); X_st = X_st.to(device)
            y_ffi = y_ffi.to(device); y_s = y_s.to(device)
            out = model(X_seq, X_st)
            ffi_pred, stress_logit = out[:, 0], out[:, 1]
            mse = F.mse_loss(ffi_pred, y_ffi)
            bce = F.binary_cross_entropy_with_logits(stress_logit, y_s, pos_weight=pos_weight)
            losses.append((LOSS_WEIGHT_REG * mse + bce).item())
            probs.append(torch.sigmoid(stress_logit).cpu().numpy())
            ys.append(y_s.cpu().numpy())
            ffis_pred.append(ffi_pred.cpu().numpy())
            ffis_true.append(y_ffi.cpu().numpy())
    probs = np.concatenate(probs); ys = np.concatenate(ys)
    ffis_pred = np.concatenate(ffis_pred); ffis_true = np.concatenate(ffis_true)
    return {
        "loss": float(np.mean(losses)),
        "auc":  float(roc_auc_score(ys, probs)),
        "ap":   float(average_precision_score(ys, probs)),
        "rmse_ffi": float(np.sqrt(np.mean((ffis_pred - ffis_true) ** 2))),
    }


def main():
    torch.manual_seed(SEED); np.random.seed(SEED)
    device = pick_device()
    print(f"Device: {device}")

    with open(ARR / "feature_spec.json") as f:
        spec = json.load(f)
    n_time = spec["n_time_features"]; n_static = spec["n_static_features"]
    print(f"  n_time_features={n_time}  n_static_features={n_static}")

    (Xtr, Str, ytr_ffi, ytr_s), (Xva, Sva, yva_ffi, yva_s), (Xte, Ste, yte_ffi, yte_s) = load_arrays()
    print(f"  train {Xtr.shape}  val {Xva.shape}  test {Xte.shape}")

    pos_weight = torch.tensor((1 - ytr_s.mean()) / max(ytr_s.mean(), 1e-6),
                              dtype=torch.float32, device=device)
    print(f"  pos_weight = {float(pos_weight):.3f}")

    tr_loader = make_loader(Xtr, Str, ytr_ffi, ytr_s, BATCH, shuffle=True)
    va_loader = make_loader(Xva, Sva, yva_ffi, yva_s, BATCH, shuffle=False)
    te_loader = make_loader(Xte, Ste, yte_ffi, yte_s, BATCH, shuffle=False)

    model = LSTMEarlyWarning(n_time, n_static).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WD)
    print(f"  model params: {sum(p.numel() for p in model.parameters()):,}")

    history = []
    best_val_auc = -np.inf
    bad_epochs = 0
    t0 = time.time()
    for epoch in range(1, EPOCHS + 1):
        model.train()
        ep_losses = []
        for X_seq, X_st, y_ffi, y_s in tr_loader:
            X_seq = X_seq.to(device); X_st = X_st.to(device)
            y_ffi = y_ffi.to(device); y_s = y_s.to(device)
            out = model(X_seq, X_st)
            ffi_pred, stress_logit = out[:, 0], out[:, 1]
            mse = F.mse_loss(ffi_pred, y_ffi)
            bce = F.binary_cross_entropy_with_logits(stress_logit, y_s, pos_weight=pos_weight)
            loss = LOSS_WEIGHT_REG * mse + bce
            opt.zero_grad(); loss.backward(); opt.step()
            ep_losses.append(loss.item())
        tr_loss = float(np.mean(ep_losses))
        val = evaluate(model, va_loader, device, pos_weight)
        history.append({
            "epoch": epoch, "train_loss": tr_loss,
            "val_loss": val["loss"], "val_auc": val["auc"],
            "val_ap": val["ap"], "val_rmse_ffi": val["rmse_ffi"],
        })
        print(f"  epoch {epoch:02d}  train {tr_loss:.4f}  val {val['loss']:.4f} "
              f"AUC {val['auc']:.4f}  AP {val['ap']:.4f}  RMSE {val['rmse_ffi']:.4f}")
        if val["auc"] > best_val_auc:
            best_val_auc = val["auc"]
            bad_epochs = 0
            torch.save({
                "state_dict": model.state_dict(),
                "n_time_features": n_time, "n_static_features": n_static,
                "epoch": epoch, "val_auc": val["auc"],
            }, CKPT)
        else:
            bad_epochs += 1
            if bad_epochs >= PATIENCE:
                print(f"  early stopping (no val-AUC improvement for {PATIENCE} epochs)")
                break
    secs = time.time() - t0

    # Load best, evaluate on test
    model.load_state_dict(torch.load(CKPT, map_location=device)["state_dict"])
    test_metrics = evaluate(model, te_loader, device, pos_weight)
    print(f"\nBest checkpoint val AUC = {best_val_auc:.4f}")
    print(f"Test metrics: AUC={test_metrics['auc']:.4f}  AP={test_metrics['ap']:.4f} "
          f"RMSE(FFI)={test_metrics['rmse_ffi']:.4f}  Loss={test_metrics['loss']:.4f}")

    with open(LOG, "w") as f:
        json.dump({
            "history": history,
            "best_val_auc": best_val_auc,
            "test": test_metrics,
            "wallclock_seconds": secs,
            "device": str(device),
            "hyperparams": {"hidden": HIDDEN, "lstm_layers": LSTM_LAYERS,
                            "dropout": DROPOUT, "loss_weight_reg": LOSS_WEIGHT_REG,
                            "lr": LR, "weight_decay": WD, "batch": BATCH,
                            "epochs_run": len(history)},
        }, f, indent=2)
    print(f"Wrote {LOG}")

    # Training curves figure
    epochs = [h["epoch"] for h in history]
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    ax = axes[0]
    ax.plot(epochs, [h["train_loss"] for h in history], label="Train loss")
    ax.plot(epochs, [h["val_loss"] for h in history], label="Val loss")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Loss (0.5·MSE + BCE)")
    ax.set_title("LSTM training loss"); ax.legend(); ax.grid(alpha=0.3)
    ax = axes[1]
    ax.plot(epochs, [h["val_auc"] for h in history], color="C2", label="Val AUC")
    ax.plot(epochs, [h["val_ap"] for h in history], color="C3", ls="--", label="Val AP")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Metric")
    ax.set_title("Validation discrimination during training"); ax.legend(); ax.grid(alpha=0.3)
    fig.tight_layout(); fig.savefig(FIG / "21_lstm_training_curves.png", dpi=200)
    plt.close(fig)
    print(f"Wrote {FIG / '21_lstm_training_curves.png'}")


# (script entry point; uncomment to re-run)
# main()


![21_lstm_training_curves.png](figures/21_lstm_training_curves.png)

_Figure produced by the cell above (saved to `figures/21_lstm_training_curves.png`)._

## 10. MC Dropout Inference

Runs 50 stochastic forward passes with dropout active at test time to draw samples from the approximate predictive posterior (Gal & Ghahramani 2016). Derives a Bayesian three-tier risk score (Low / Medium / High) from posterior quantiles.

In [ ]:
"""
MC Dropout inference for the trained LSTM early-warning model.

Gal & Ghahramani (2016) show that a neural network trained with dropout, with
dropout left active at inference, is mathematically equivalent to a variational
approximation to a deep Gaussian process. K stochastic forward passes give K
samples from the approximate posterior over predictions; their mean is the
posterior point estimate and their dispersion is the (epistemic) uncertainty.

This script:
  1. loads the checkpoint and the test arrays
  2. with dropout layers in train() mode and grad disabled, runs K=50 forward passes
  3. assembles per-row predictive distributions for FFI_{t+3} (regression) and
     for the stress-event probability (classification)
  4. derives Low / Medium / High risk tiers from the posterior:
        High   = upper 95% credible bound of p(stress) > 0.5
        Medium = posterior mean of p(stress)         > 0.5
        Low    = otherwise
  5. saves outputs for downstream evaluation against the baselines

Outputs:
  early_warning/lstm_arrays/test_predictions.npz
    keys: ffi_mean, ffi_std, ffi_low95, ffi_high95,
          stress_mean, stress_std, stress_low95, stress_high95,
          risk_tier (int8: 0=Low, 1=Med, 2=High)
  early_warning/lstm_arrays/test_predictions_summary.json
"""
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset

from lstm_train import LSTMEarlyWarning, pick_device

ROOT = Path(__file__).resolve().parent.parent
ARR = ROOT / "early_warning" / "lstm_arrays"
CKPT = ARR / "lstm_checkpoint.pt"
OUT_NPZ = ARR / "test_predictions.npz"
OUT_JSON = ARR / "test_predictions_summary.json"

K_MC = 50
BATCH = 512


def main():
    device = pick_device()
    print(f"Device: {device}")

    # Reproducibility for the MC samples (do not affect a re-run after retraining)
    torch.manual_seed(123)
    np.random.seed(123)

    with open(ARR / "feature_spec.json") as f:
        spec = json.load(f)
    n_time = spec["n_time_features"]; n_static = spec["n_static_features"]

    ckpt = torch.load(CKPT, map_location=device)
    model = LSTMEarlyWarning(n_time, n_static).to(device)
    model.load_state_dict(ckpt["state_dict"])

    # Critical: leave model in train() mode so dropout layers stay active at
    # inference. PyTorch's nn.LSTM dropout is *only* applied during train()
    # mode AND only between stacked layers (which is exactly what we have, 2-layer).
    model.train()

    X_seq = torch.from_numpy(np.load(ARR / "test_X_seq.npy"))
    X_st  = torch.from_numpy(np.load(ARR / "test_X_static.npy"))
    y_ffi = np.load(ARR / "test_y_ffi.npy")
    y_s   = np.load(ARR / "test_y_stress.npy")
    print(f"  test rows: {len(y_s):,}")

    loader = DataLoader(
        TensorDataset(X_seq, X_st), batch_size=BATCH, shuffle=False, num_workers=0,
    )

    # Pre-allocate per-sample collections to save memory
    N = len(y_s)
    ffi_samples = np.zeros((K_MC, N), dtype=np.float32)
    p_samples   = np.zeros((K_MC, N), dtype=np.float32)

    print(f"Running {K_MC} MC-Dropout forward passes (dropout active)...")
    with torch.no_grad():
        for k in range(K_MC):
            ptr = 0
            for X1, X2 in loader:
                X1 = X1.to(device); X2 = X2.to(device)
                out = model(X1, X2)
                b = out.shape[0]
                ffi_samples[k, ptr:ptr + b] = out[:, 0].cpu().numpy()
                p_samples[k,   ptr:ptr + b] = torch.sigmoid(out[:, 1]).cpu().numpy()
                ptr += b
            if (k + 1) % 10 == 0:
                print(f"  sample {k+1}/{K_MC} done")

    # Posterior summaries
    ffi_mean  = ffi_samples.mean(axis=0)
    ffi_std   = ffi_samples.std(axis=0)
    ffi_low95 = np.percentile(ffi_samples, 2.5,  axis=0)
    ffi_high95= np.percentile(ffi_samples, 97.5, axis=0)

    p_mean   = p_samples.mean(axis=0)
    p_std    = p_samples.std(axis=0)
    p_low95  = np.percentile(p_samples, 2.5,  axis=0)
    p_high95 = np.percentile(p_samples, 97.5, axis=0)

    # Risk-tier mapping that *actually uses* the posterior uncertainty.
    # The naive "High = upper > 0.5, Medium = mean > 0.5" rule collapses to two
    # tiers because mean <= upper always, so we cannot have Medium with the
    # upper bound below the cutoff. The Bayesian three-tier rule we want is:
    #
    #   High    = lower bound > 0.5     (confidently in stress)
    #   Medium  = mean        > 0.5  but lower bound <= 0.5   (likely-but-uncertain)
    #   Low     = mean        <= 0.5
    #
    # Then "Medium" precisely captures predictions whose posterior credible
    # interval straddles the decision boundary, which is the right operational
    # definition of "uncertain" for policy triage.
    tier = np.where(
        p_low95 > 0.5, 2,
        np.where(p_mean > 0.5, 1, 0),
    ).astype(np.int8)

    counts = {f"tier_{t}": int((tier == t).sum()) for t in [0, 1, 2]}
    pos = y_s == 1
    capture = {
        f"tier_{t}_positive_share": float(((tier == t) & pos).sum() / max(pos.sum(), 1))
        for t in [0, 1, 2]
    }
    purity = {
        f"tier_{t}_purity": (
            float(pos[(tier == t)].mean()) if (tier == t).any() else None
        )
        for t in [0, 1, 2]
    }

    print(f"  Tier counts: {counts}")
    print(f"  Positives captured by tier: {capture}")
    print(f"  Tier purity (= positive rate within tier): {purity}")

    np.savez(
        OUT_NPZ,
        ffi_mean=ffi_mean, ffi_std=ffi_std,
        ffi_low95=ffi_low95, ffi_high95=ffi_high95,
        stress_mean=p_mean, stress_std=p_std,
        stress_low95=p_low95, stress_high95=p_high95,
        risk_tier=tier,
        y_ffi=y_ffi, y_stress=y_s,
    )
    print(f"Wrote {OUT_NPZ}")

    with open(OUT_JSON, "w") as f:
        json.dump({
            "K_MC": K_MC,
            "device": str(device),
            "tier_counts": counts,
            "tier_capture_of_positives": capture,
            "tier_purity": purity,
            "mean_predictive_std_ffi": float(ffi_std.mean()),
            "mean_predictive_std_p":   float(p_std.mean()),
        }, f, indent=2)
    print(f"Wrote {OUT_JSON}")


# (script entry point; uncomment to re-run)
# main()


## 11. Baselines (naive, NGBoost, Kalman)

Three reference models the LSTM must beat: naive carry-forward; NGBoost (probabilistic gradient boosting with natural-gradient descent); and per-household Kalman state-space filter (local-level + AR(1) Unobserved Components).

In [ ]:
"""
Three baselines the LSTM must beat:

  (1) Naive carry-forward: y_hat_{t+3} = y_t. The brick wall every time-series
      model has to clear.

  (2) NGBoost (Duan et al. 2020): natural-gradient boosting that natively
      predicts a parametric distribution (Normal mean+std for regression,
      Bernoulli for classification) rather than a point estimate. We fit on
      lagged-window features (the same 12-month window used by the LSTM,
      flattened to a vector + static features).

  (3) Kalman filter (Unobserved Components state-space model): treats
      ffi_t as a local level + AR(1) process. Forecasts 3 steps ahead per
      household. Provides a principled mean+variance forecast.

For each baseline we produce predictions for the held-out test households.
The regression target is FFI_{t+3} (in z-score units). The binary target is
the same y_stress (max ffi in next 3 months > tau).

Outputs:
  early_warning/lstm_arrays/baseline_naive.npz
  early_warning/lstm_arrays/baseline_ngboost.npz
  early_warning/lstm_arrays/baseline_kalman.npz
  early_warning/lstm_arrays/baselines_summary.json
"""
from __future__ import annotations

import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from ngboost import NGBClassifier, NGBRegressor
from ngboost.distns import Bernoulli, Normal
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.tree import DecisionTreeRegressor

warnings.filterwarnings("ignore", category=FutureWarning)

ROOT = Path(__file__).resolve().parent.parent
ARR = ROOT / "early_warning" / "lstm_arrays"
PANEL = ROOT / "early_warning" / "panel_with_ffi.parquet"
CAL = ROOT / "early_warning" / "ffi_calibration.json"
OUT_SUMMARY = ARR / "baselines_summary.json"

RNG = 42


def load_split_arrays():
    def _load(split):
        return {
            "X_seq":   np.load(ARR / f"{split}_X_seq.npy"),
            "X_st":    np.load(ARR / f"{split}_X_static.npy"),
            "y_ffi":   np.load(ARR / f"{split}_y_ffi.npy"),
            "y_str":   np.load(ARR / f"{split}_y_stress.npy"),
            "hh_id":   np.load(ARR / f"{split}_hh_id.npy", allow_pickle=True),
        }
    return _load("train"), _load("val"), _load("test")


def baseline_naive(test: dict, spec: dict, tau: float):
    """y_hat_{t+3} = ffi_t  (carry forward the latest observed FFI)."""
    print("\n--- Baseline 1: naive carry-forward ---")
    feat_idx = spec["time_features"].index("ffi_t")
    # ffi_t at the LAST history month (column-major position L-1 of the sequence)
    last_ffi = test["X_seq"][:, -1, feat_idx]
    # The time features were z-scored across the training pool. ffi_t was
    # already in z-units before scaling, so "carry-forward" predictions are
    # given in the same z-scaled units as y_ffi. To compare on the original
    # FFI scale we need to invert the per-feature scaler.
    s = spec["scalers"]["ffi_t"]
    ffi_pred = last_ffi * s["std"] + s["mean"]
    # Stress probability: 1 if ffi_pred > tau, else 0; for AUC we use the raw
    # forecast as a score (higher -> more likely stress).
    stress_score = ffi_pred
    auc = roc_auc_score(test["y_str"], stress_score)
    ap  = average_precision_score(test["y_str"], stress_score)
    rmse = float(np.sqrt(np.mean((ffi_pred - test["y_ffi"]) ** 2)))
    print(f"  AUC {auc:.4f}  AP {ap:.4f}  RMSE {rmse:.4f}")
    np.savez(ARR / "baseline_naive.npz",
             ffi_pred=ffi_pred, stress_score=stress_score,
             y_ffi=test["y_ffi"], y_stress=test["y_str"])
    return {"auc": float(auc), "ap": float(ap), "rmse_ffi": rmse}


def _flatten_sequence_features(seq: np.ndarray, static: np.ndarray) -> np.ndarray:
    """For NGBoost: turn (N, L, F_t) + (N, F_s) into a single (N, L*F_t + F_s) matrix."""
    n, L, F = seq.shape
    return np.concatenate([seq.reshape(n, L * F), static], axis=1).astype(np.float32)


def baseline_ngboost(train, val, test, spec):
    """NGBoost: probabilistic gradient boosting that learns p(y | x) = N(mu, sigma).

    For the binary head we use NGBClassifier with Bernoulli."""
    print("\n--- Baseline 2: NGBoost on lagged-window features ---")
    Xtr = _flatten_sequence_features(train["X_seq"], train["X_st"])
    Xva = _flatten_sequence_features(val["X_seq"],   val["X_st"])
    Xte = _flatten_sequence_features(test["X_seq"],  test["X_st"])
    print(f"  flattened: train {Xtr.shape}  val {Xva.shape}  test {Xte.shape}")

    # Subsample training to keep wall-clock manageable; NGBoost is single-threaded
    # and 295k x 204 features is ~2h at default settings. 80k rows is plenty.
    rng = np.random.default_rng(RNG)
    sub = rng.choice(len(Xtr), size=min(80_000, len(Xtr)), replace=False)
    Xtr_s = Xtr[sub]; ytr_ffi_s = train["y_ffi"][sub]; ytr_str_s = train["y_str"][sub]

    base = DecisionTreeRegressor(max_depth=5, min_samples_leaf=50)
    t0 = time.time()
    reg = NGBRegressor(Dist=Normal, Base=base, n_estimators=300,
                       learning_rate=0.04, verbose=False, random_state=RNG)
    reg.fit(Xtr_s, ytr_ffi_s, X_val=Xva, Y_val=val["y_ffi"], early_stopping_rounds=20)
    print(f"  NGBRegressor fit done in {time.time()-t0:.1f}s")

    t0 = time.time()
    clf = NGBClassifier(Dist=Bernoulli, Base=base, n_estimators=300,
                        learning_rate=0.04, verbose=False, random_state=RNG)
    clf.fit(Xtr_s, ytr_str_s.astype(int), X_val=Xva, Y_val=val["y_str"].astype(int),
            early_stopping_rounds=20)
    print(f"  NGBClassifier fit done in {time.time()-t0:.1f}s")

    # Predict on test
    ffi_dist = reg.pred_dist(Xte)
    ffi_mean = ffi_dist.mean(); ffi_std = ffi_dist.std()
    rmse = float(np.sqrt(np.mean((ffi_mean - test["y_ffi"]) ** 2)))

    # NGBClassifier supports the sklearn predict_proba(X) -> (n, n_classes).
    p_mean = clf.predict_proba(Xte)[:, 1]

    auc = roc_auc_score(test["y_str"], p_mean)
    ap  = average_precision_score(test["y_str"], p_mean)
    print(f"  AUC {auc:.4f}  AP {ap:.4f}  RMSE {rmse:.4f}")

    np.savez(ARR / "baseline_ngboost.npz",
             ffi_mean=ffi_mean.astype(np.float32),
             ffi_std=ffi_std.astype(np.float32),
             stress_mean=p_mean.astype(np.float32),
             y_ffi=test["y_ffi"], y_stress=test["y_str"])
    return {"auc": float(auc), "ap": float(ap), "rmse_ffi": rmse,
            "mean_pred_std_ffi": float(np.mean(ffi_std))}


def baseline_kalman(panel: pd.DataFrame, test: dict, spec: dict, tau_scaled: float):
    """Per-household Kalman filter / Unobserved Components on ffi_t.

    OPTIMISED: fit one state-space model per household on its FULL history
    (months 1..21, the latest training point shared across all test rows for
    that hh), then use the fitted filter to extract dynamic 3-step-ahead
    forecasts from each prediction-time prefix t in [12..21]. That cuts the
    per-row fitting cost from 10 fits/hh to 1 fit/hh, taking total wall-clock
    from ~60 minutes to ~10 minutes.
    """
    print("\n--- Baseline 3: Kalman / UnobservedComponents per household (1 fit / hh) ---")
    from collections import defaultdict
    from statsmodels.tsa.statespace.structural import UnobservedComponents

    s = spec["scalers"]["ffi_t"]
    history_len = spec["history_len"]
    horizon = spec["horizon"]
    test_hh_unique = np.unique(test["hh_id"])

    # Time-varying FFI series per household, on the original z-score scale
    p = panel.set_index(["hh_id", "month"]).sort_index()
    ffi_series = {hh: p.loc[hh, "ffi_t"].values for hh in test_hh_unique}

    # Build hh -> list of (row_idx, t) where t is the month of the last history obs
    hh_to_rows = defaultdict(list)
    for i, hh in enumerate(test["hh_id"]):
        hh_to_rows[hh].append(i)
    for hh in hh_to_rows:
        hh_to_rows[hh] = [(idx, history_len + off) for off, idx in enumerate(hh_to_rows[hh])]

    # We fit on the longest history available (max t per hh in the test set),
    # which equals TOTAL_MONTHS - HORIZON = 21 for almost all hh by construction.
    N = len(test["hh_id"])
    ffi_pred = np.zeros(N, dtype=np.float32)
    stress_score = np.zeros(N, dtype=np.float32)
    fail_count = 0
    t0 = time.time()
    n_hh = len(hh_to_rows)

    for i_hh, (hh, row_list) in enumerate(hh_to_rows.items()):
        series = ffi_series[hh]
        max_t = max(t for _, t in row_list)        # longest history we need
        fit_history = series[:max_t]
        try:
            mod = UnobservedComponents(fit_history, level="local level", autoregressive=1)
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                res = mod.fit(disp=False, maxiter=30, method="lbfgs")
            # For each prediction-time prefix t, get dynamic 3-step-ahead forecasts
            for row_idx, t in row_list:
                # res.get_prediction(start=t, end=t+horizon-1, dynamic=True) gives
                # forecasts conditioned on data up to (start - 1).
                pred = res.get_prediction(start=t, end=t + horizon - 1, dynamic=True)
                mean = pred.predicted_mean
                # Per-prediction-time 3-step horizon prediction
                ffi_pred[row_idx] = mean[-1]
                stress_score[row_idx] = max(mean)
        except Exception:
            for row_idx, t in row_list:
                ffi_pred[row_idx] = series[t - 1] if t - 1 < len(series) else 0.0
                stress_score[row_idx] = ffi_pred[row_idx]
            fail_count += 1
        if (i_hh + 1) % 500 == 0:
            elapsed = time.time() - t0
            rate = (i_hh + 1) / elapsed
            eta = (n_hh - i_hh - 1) / rate
            print(f"  fitted {i_hh+1:,}/{n_hh:,} hh  ({rate:.1f} hh/s,  ETA {eta/60:.1f} min)")
    print(f"  Kalman finished in {time.time()-t0:.1f}s  (fallbacks: {fail_count})")

    # Both forecasts are on the original FFI scale; convert to z-scale to match y_ffi.
    # Fallback rows may contain NaN if the underlying series had NaN at the prefix
    # index; we substitute 0 (the z-scored pool mean) before computing metrics.
    ffi_pred_z = np.nan_to_num((ffi_pred - s["mean"]) / s["std"], nan=0.0, posinf=0.0, neginf=0.0)
    stress_score_z = np.nan_to_num((stress_score - s["mean"]) / s["std"], nan=0.0, posinf=0.0, neginf=0.0)

    rmse = float(np.sqrt(np.mean((ffi_pred_z - test["y_ffi"]) ** 2)))
    auc = roc_auc_score(test["y_str"], stress_score_z)
    ap  = average_precision_score(test["y_str"], stress_score_z)
    print(f"  AUC {auc:.4f}  AP {ap:.4f}  RMSE {rmse:.4f}")
    np.savez(ARR / "baseline_kalman.npz",
             ffi_pred=ffi_pred_z, stress_score=stress_score_z,
             y_ffi=test["y_ffi"], y_stress=test["y_str"])
    return {"auc": float(auc), "ap": float(ap), "rmse_ffi": rmse,
            "fallbacks": fail_count}


def main():
    with open(ARR / "feature_spec.json") as f:
        spec = json.load(f)
    with open(CAL) as f:
        ffi_cal = json.load(f)
    tau = float(ffi_cal["tau_75th_pct"])
    print(f"FFI threshold tau = {tau:.4f} (z-score scale)")

    train, val, test = load_split_arrays()
    print(f"Splits: train {len(train['y_ffi']):,}  val {len(val['y_ffi']):,}  test {len(test['y_ffi']):,}")

    results = {}

    # Re-use any already-saved baseline outputs to make this script restart-safe
    # (NGBoost is the bottleneck; we don't want to refit it on a re-run).
    def _reuse(name: str, runner):
        npz_path = ARR / f"baseline_{name}.npz"
        if npz_path.exists():
            print(f"\n--- {name}: re-using cached {npz_path.name} ---")
            d = np.load(npz_path)
            score_key = "stress_score" if name != "ngboost" else "stress_mean"
            stress = d[score_key]
            rmse = float(np.sqrt(np.mean((d["ffi_pred" if name != "ngboost" else "ffi_mean"]
                                          - d["y_ffi"]) ** 2)))
            auc = float(roc_auc_score(d["y_stress"], stress))
            ap  = float(average_precision_score(d["y_stress"], stress))
            print(f"  AUC {auc:.4f}  AP {ap:.4f}  RMSE {rmse:.4f}")
            return {"auc": auc, "ap": ap, "rmse_ffi": rmse, "reused": True}
        return runner()

    results["naive"]   = _reuse("naive",   lambda: baseline_naive(test, spec, tau))
    results["ngboost"] = _reuse("ngboost", lambda: baseline_ngboost(train, val, test, spec))
    panel = pd.read_parquet(PANEL)
    results["kalman"]  = _reuse("kalman",  lambda: baseline_kalman(panel, test, spec, tau))

    with open(OUT_SUMMARY, "w") as f:
        json.dump(results, f, indent=2)
    print(f"\nWrote {OUT_SUMMARY}")
    print("Summary:")
    for k, v in results.items():
        print(f"  {k:10s} AUC {v['auc']:.4f}  AP {v['ap']:.4f}  RMSE {v['rmse_ffi']:.4f}")


# (script entry point; uncomment to re-run)
# main()


## 12. Evaluation, Figures, and Risk Tiers

ROC and PR curves across all four models, reliability diagram for the probabilistic models, per-household uncertainty band illustrations, and the risk-tier triage table.

In [ ]:
"""
Assemble the held-out test-set evaluation across all four early-warning models
(LSTM + MC Dropout, naive, NGBoost, Kalman), and produce the figures and
summary table that feed §4.7 / §4.8 of the report.

Metrics:
  - ROC-AUC and Average Precision for the binary stress event
  - RMSE for the 3-month-ahead FFI regression
  - Expected Calibration Error (ECE, 10 equal-frequency bins) for probabilistic models
  - 95% credible-interval coverage (LSTM and NGBoost only)

Figures:
  22_ew_roc_pr.png            ROC and PR curves for all four models
  23_ew_calibration.png       Reliability diagram for LSTM and NGBoost
  24_ew_uncertainty_examples.png   Sample household predictions with credible intervals
  25_ew_risk_tier.png         Risk-tier counts and per-tier positive rate
  26_ew_coverage.png          Empirical coverage vs nominal coverage (LSTM)

Outputs:
  early_warning/lstm_arrays/ew_summary.json   final numbers for the report table
"""
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import (
    average_precision_score, precision_recall_curve, roc_auc_score, roc_curve,
)

ROOT = Path(__file__).resolve().parent.parent
ARR = ROOT / "early_warning" / "lstm_arrays"
FIG = ROOT / "figures"
OUT = ARR / "ew_summary.json"


def ece_score(y_true: np.ndarray, p: np.ndarray, n_bins: int = 10) -> float:
    """Expected Calibration Error with equal-frequency bins."""
    order = np.argsort(p)
    y_sorted = y_true[order]; p_sorted = p[order]
    bins = np.array_split(np.arange(len(p)), n_bins)
    ece = 0.0
    for b in bins:
        if len(b) == 0:
            continue
        mean_p = p_sorted[b].mean()
        mean_y = y_sorted[b].mean()
        ece += (len(b) / len(p)) * abs(mean_p - mean_y)
    return float(ece)


def credible_coverage(y_ffi: np.ndarray, lo: np.ndarray, hi: np.ndarray) -> float:
    return float(np.mean((y_ffi >= lo) & (y_ffi <= hi)))


def load_all():
    """Load all model predictions for the test set, harmonising key names."""
    out = {}

    # LSTM (full posterior)
    lstm = np.load(ARR / "test_predictions.npz")
    out["lstm"] = {
        "ffi_mean": lstm["ffi_mean"], "ffi_low": lstm["ffi_low95"], "ffi_high": lstm["ffi_high95"],
        "stress_mean": lstm["stress_mean"], "stress_low": lstm["stress_low95"], "stress_high": lstm["stress_high95"],
        "risk_tier": lstm["risk_tier"],
        "y_ffi": lstm["y_ffi"], "y_stress": lstm["y_stress"],
    }
    # Naive
    nv = np.load(ARR / "baseline_naive.npz")
    out["naive"] = {
        "ffi_mean": nv["ffi_pred"], "stress_mean": nv["stress_score"],
        "y_ffi": nv["y_ffi"], "y_stress": nv["y_stress"],
    }
    # NGBoost
    if (ARR / "baseline_ngboost.npz").exists():
        ng = np.load(ARR / "baseline_ngboost.npz")
        # NGBoost stress mean is calibrated probability already in [0,1]
        out["ngboost"] = {
            "ffi_mean": ng["ffi_mean"], "ffi_std": ng["ffi_std"],
            "stress_mean": ng["stress_mean"],
            "y_ffi": ng["y_ffi"], "y_stress": ng["y_stress"],
        }
        # 95% credible interval from the parametric Normal
        out["ngboost"]["ffi_low"]  = ng["ffi_mean"] - 1.96 * ng["ffi_std"]
        out["ngboost"]["ffi_high"] = ng["ffi_mean"] + 1.96 * ng["ffi_std"]
    # Kalman
    if (ARR / "baseline_kalman.npz").exists():
        kl = np.load(ARR / "baseline_kalman.npz")
        out["kalman"] = {
            "ffi_mean": kl["ffi_pred"], "stress_mean": kl["stress_score"],
            "y_ffi": kl["y_ffi"], "y_stress": kl["y_stress"],
        }

    return out


def metrics_for(name: str, data: dict) -> dict:
    y_s = data["y_stress"]
    y_f = data["y_ffi"]
    p = data["stress_mean"]
    # For naive / kalman the "score" is the predicted FFI on the z-scale,
    # not a probability. ROC-AUC is scale-invariant so this is OK for AUC/AP,
    # but ECE is not meaningful unless we map it to [0,1]. Skip ECE for those.
    auc = roc_auc_score(y_s, p)
    ap = average_precision_score(y_s, p)
    rmse = float(np.sqrt(np.mean((data["ffi_mean"] - y_f) ** 2)))
    out = {"auc": float(auc), "ap": float(ap), "rmse_ffi": rmse}
    if name in ("lstm", "ngboost"):
        out["ece"] = ece_score(y_s, p)
        out["coverage_95"] = credible_coverage(y_f, data["ffi_low"], data["ffi_high"])
    return out


def figure_roc_pr(all_data: dict):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    order = ["naive", "kalman", "ngboost", "lstm"]
    colors = {"naive": "gray", "kalman": "C0", "ngboost": "C2", "lstm": "C3"}
    labels = {"naive": "Naive carry-forward", "kalman": "Kalman UC",
              "ngboost": "NGBoost (lagged)", "lstm": "LSTM + MC Dropout"}
    ax = axes[0]
    for k in order:
        if k not in all_data: continue
        d = all_data[k]
        fpr, tpr, _ = roc_curve(d["y_stress"], d["stress_mean"])
        auc = roc_auc_score(d["y_stress"], d["stress_mean"])
        ax.plot(fpr, tpr, color=colors[k], lw=1.8, label=f"{labels[k]} (AUC={auc:.3f})")
    ax.plot([0, 1], [0, 1], "k--", lw=0.8)
    ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
    ax.set_title("ROC — early-warning, 3-month horizon")
    ax.legend(loc="lower right"); ax.grid(alpha=0.3)
    ax = axes[1]
    for k in order:
        if k not in all_data: continue
        d = all_data[k]
        pr, rc, _ = precision_recall_curve(d["y_stress"], d["stress_mean"])
        ap = average_precision_score(d["y_stress"], d["stress_mean"])
        ax.plot(rc, pr, color=colors[k], lw=1.8, label=f"{labels[k]} (AP={ap:.3f})")
    base = float(all_data["lstm"]["y_stress"].mean())
    ax.axhline(base, color="k", ls="--", lw=0.8, label=f"baseline = {base:.3f}")
    ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
    ax.set_title("Precision-Recall — early-warning, 3-month horizon")
    ax.legend(loc="lower left"); ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(FIG / "22_ew_roc_pr.png", dpi=200)
    plt.close(fig)


def figure_calibration(all_data: dict):
    from sklearn.calibration import calibration_curve
    fig, ax = plt.subplots(figsize=(6, 5))
    for k, color, label in [
        ("lstm", "C3", "LSTM + MC Dropout"),
        ("ngboost", "C2", "NGBoost"),
    ]:
        if k not in all_data: continue
        d = all_data[k]
        p = np.clip(d["stress_mean"], 1e-6, 1 - 1e-6)
        frac, pred = calibration_curve(d["y_stress"], p,
                                       n_bins=10, strategy="quantile")
        ece = ece_score(d["y_stress"], p)
        ax.plot(pred, frac, "o-", color=color, label=f"{label} (ECE={ece:.3f})")
    ax.plot([0, 1], [0, 1], "k--", lw=0.8, label="perfectly calibrated")
    ax.set_xlabel("Mean predicted probability")
    ax.set_ylabel("Empirical fraction positive")
    ax.set_title("Reliability diagram — stress probability (10 quantile bins)")
    ax.legend(); ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(FIG / "23_ew_calibration.png", dpi=200)
    plt.close(fig)


def figure_uncertainty_examples(all_data: dict):
    """For six representative test rows, plot the predicted FFI_{t+3} with its
    LSTM credible interval and the true value."""
    d = all_data["lstm"]
    rng = np.random.default_rng(0)
    # Choose 3 rows where LSTM is confident-correct, 3 where it's uncertain
    p = d["stress_mean"]; spread = d["stress_high"] - d["stress_low"]
    confident = np.where((p > 0.9) | (p < 0.1))[0]
    uncertain = np.where((p > 0.3) & (p < 0.7))[0]
    pick = np.concatenate([
        rng.choice(confident, size=min(3, len(confident)), replace=False),
        rng.choice(uncertain, size=min(3, len(uncertain)), replace=False),
    ])
    titles = ["Confident A", "Confident B", "Confident C",
              "Uncertain A", "Uncertain B", "Uncertain C"]
    fig, axes = plt.subplots(2, 3, figsize=(13, 6))
    for ax, idx, ttl in zip(axes.flat, pick, titles):
        # Plot the 50 MC posterior samples of stress probability around mean
        mean_p = float(p[idx]); lo = float(d["stress_low"][idx]); hi = float(d["stress_high"][idx])
        y_true = int(d["y_stress"][idx]); ffi_true = float(d["y_ffi"][idx])
        ffi_pred = float(d["ffi_mean"][idx]); ffi_lo = float(d["ffi_low"][idx]); ffi_hi = float(d["ffi_high"][idx])

        # Two stacked sub-plots in this cell: FFI band + stress prob bar
        ax.errorbar([0], [ffi_pred], yerr=[[ffi_pred - ffi_lo], [ffi_hi - ffi_pred]],
                    fmt="o", color="C0", capsize=6, label=f"Predicted FFI ± 95% CI")
        ax.scatter([0], [ffi_true], color="red", s=60, zorder=5, label=f"True FFI")
        ax.bar([1], [mean_p], yerr=[[mean_p - lo], [hi - mean_p]],
               width=0.4, color="C3", alpha=0.6, capsize=6, label=f"Predicted p(stress) ± 95% CI")
        ax.axhline(0.5, xmin=0.4, xmax=1.0, color="gray", ls="--", lw=0.7)
        ax.text(1, 1.02, f"true y = {y_true}", ha="center", va="bottom", fontsize=8, color="red")
        ax.set_xticks([0, 1]); ax.set_xticklabels(["FFI$_{t+3}$", "p(stress)"])
        ax.set_ylim(-3, 3.5)
        ax.set_title(ttl, fontsize=10)
        ax.grid(alpha=0.3, axis="y")
    axes[0, 0].legend(loc="upper right", fontsize=7)
    fig.suptitle("LSTM posterior predictions on individual test rows: "
                 "confident correct vs uncertain", y=1.02)
    fig.tight_layout()
    fig.savefig(FIG / "24_ew_uncertainty_examples.png", dpi=200, bbox_inches="tight")
    plt.close(fig)


def figure_risk_tier(all_data: dict):
    d = all_data["lstm"]
    tier = d["risk_tier"]; y = d["y_stress"]
    names = ["Low", "Medium", "High"]
    counts = [int((tier == i).sum()) for i in range(3)]
    purity = [float(y[tier == i].mean()) if (tier == i).any() else 0.0 for i in range(3)]
    capture = [float(((tier == i) & (y == 1)).sum() / max(y.sum(), 1)) for i in range(3)]
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    ax = axes[0]
    ax.bar(names, counts, color=["#2ca02c", "#ff7f0e", "#d62728"])
    for n, c in zip(names, counts):
        ax.text(n, c, f"{c:,}", ha="center", va="bottom", fontsize=10)
    ax.set_ylabel("Test rows")
    ax.set_title("Risk tier population (LSTM + MC Dropout)")
    ax.grid(alpha=0.3, axis="y")
    ax = axes[1]
    xs = np.arange(3); w = 0.35
    ax.bar(xs - w/2, purity, w, color="C3", label="Purity (positive rate within tier)")
    ax.bar(xs + w/2, capture, w, color="C0", label="Recall (share of positives captured)")
    for x, p in zip(xs, purity):
        ax.text(x - w/2, p, f"{p:.2f}", ha="center", va="bottom", fontsize=9)
    for x, c in zip(xs, capture):
        ax.text(x + w/2, c, f"{c:.2f}", ha="center", va="bottom", fontsize=9)
    ax.set_xticks(xs); ax.set_xticklabels(names)
    ax.set_ylim(0, 1.1); ax.set_ylabel("Share")
    ax.set_title("Per-tier purity and recall"); ax.legend(); ax.grid(alpha=0.3, axis="y")
    fig.tight_layout()
    fig.savefig(FIG / "25_ew_risk_tier.png", dpi=200)
    plt.close(fig)


def figure_coverage(all_data: dict):
    """For LSTM, plot empirical coverage of predictive intervals at every nominal
    level from 5% to 95%."""
    # Reconstruct intervals from quantiles of the saved 95% CI alone is not
    # enough; we approximate by treating the predicted (low95, high95) as the
    # outer envelope and the mean as the centre, and plot the *one* point we
    # have plus the 50% interval derived from (mean ± 0.67 * sigma) where sigma
    # is implied by the 95% CI half-width.
    d = all_data["lstm"]
    sigma = (d["ffi_high"] - d["ffi_low"]) / (2 * 1.96)
    nominal = np.linspace(0.1, 0.95, 18)
    coverages = []
    for nl in nominal:
        z = abs(np.quantile(np.random.standard_normal(20_000), (1 + nl) / 2))
        lo = d["ffi_mean"] - z * sigma
        hi = d["ffi_mean"] + z * sigma
        coverages.append(float(np.mean((d["y_ffi"] >= lo) & (d["y_ffi"] <= hi))))
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.plot(nominal, coverages, "o-", color="C3", label="LSTM (MC Dropout, Gaussian fit)")
    ax.plot([0, 1], [0, 1], "k--", lw=0.8, label="perfect coverage")
    ax.set_xlabel("Nominal coverage")
    ax.set_ylabel("Empirical coverage")
    ax.set_title("Predictive-interval coverage diagnostic")
    ax.legend(); ax.grid(alpha=0.3); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    fig.tight_layout()
    fig.savefig(FIG / "26_ew_coverage.png", dpi=200)
    plt.close(fig)


def main():
    all_data = load_all()
    summary = {}
    for name, data in all_data.items():
        summary[name] = metrics_for(name, data)
        print(f"  {name:10s}  {summary[name]}")

    # Figures
    figure_roc_pr(all_data)
    figure_calibration(all_data)
    figure_uncertainty_examples(all_data)
    figure_risk_tier(all_data)
    figure_coverage(all_data)

    with open(OUT, "w") as f:
        json.dump(summary, f, indent=2)
    print(f"\nWrote {OUT}")


# (script entry point; uncomment to re-run)
# main()


![22_ew_roc_pr.png](figures/22_ew_roc_pr.png)

_Figure produced by the cell above (saved to `figures/22_ew_roc_pr.png`)._

## 13. Patching the Report

Convenience script that reads `ew_summary.json` + the MC Dropout tier summary and substitutes the corresponding numerical placeholders in `report/main.tex`.

In [ ]:
"""
Patch the report's §4 results table (and tier table) with the actual numbers
produced by evaluate.py. Reads early_warning/lstm_arrays/ew_summary.json and
early_warning/lstm_arrays/test_predictions_summary.json and substitutes the
[NAIVE_AUC], [LSTM_ECE], etc. placeholders in report/main.tex.

Run AFTER baselines.py and evaluate.py have completed successfully.
"""
from __future__ import annotations

import json
import re
from pathlib import Path

ROOT = Path(__file__).resolve().parent.parent
SUM = ROOT / "early_warning" / "lstm_arrays" / "ew_summary.json"
TIER = ROOT / "early_warning" / "lstm_arrays" / "test_predictions_summary.json"
TEX = ROOT / "report" / "main.tex"


def fmt(x, decimals=4):
    if x is None:
        return "---"
    return f"{x:.{decimals}f}"


def main():
    with open(SUM) as f:
        s = json.load(f)
    with open(TIER) as f:
        t = json.load(f)
    print("Loaded:", list(s.keys()))

    subs = {
        "NAIVE_AUC":   fmt(s["naive"]["auc"], 4),
        "NAIVE_AP":    fmt(s["naive"]["ap"], 4),
        "NAIVE_RMSE":  fmt(s["naive"]["rmse_ffi"], 4),
        "KALMAN_AUC":  fmt(s.get("kalman", {}).get("auc"), 4),
        "KALMAN_AP":   fmt(s.get("kalman", {}).get("ap"), 4),
        "KALMAN_RMSE": fmt(s.get("kalman", {}).get("rmse_ffi"), 4),
        "NGB_AUC":     fmt(s.get("ngboost", {}).get("auc"), 4),
        "NGB_AP":      fmt(s.get("ngboost", {}).get("ap"), 4),
        "NGB_RMSE":    fmt(s.get("ngboost", {}).get("rmse_ffi"), 4),
        "NGB_ECE":     fmt(s.get("ngboost", {}).get("ece"), 4),
        "LSTM_ECE":    fmt(s.get("lstm", {}).get("ece"), 4),
        "LSTM_AUC":    fmt(s.get("lstm", {}).get("auc"), 4),
        "MED_PUR":     fmt((t["tier_purity"].get("tier_1_purity") or 0) * 100, 1),
        "HIGH_PUR":    fmt((t["tier_purity"].get("tier_2_purity") or 0) * 100, 1),
    }
    print("Substitutions:")
    for k, v in subs.items():
        print(f"  {k:14s} -> {v}")

    # The LaTeX source uses backslash-escaped underscores inside \texttt{...},
    # so the literal placeholder to match is e.g. \texttt{[NAIVE\_AUC]}.
    text = TEX.read_text()
    for k, v in subs.items():
        k_tex = k.replace("_", r"\_")  # match the escaped form in the .tex source
        text = text.replace(f"\\texttt{{[{k_tex}]}}", v)
    TEX.write_text(text)
    print(f"Patched {TEX}")


# (script entry point; uncomment to re-run)
# main()
